# 운수종사자 교통사고 위험 예측 - 종합 피처 엔지니어링

## 목표
- PrimaryKey 기준으로 A/B 검사 로그를 driver-level 피처로 변환
- 각 검사별 설명에 맞는 의미있는 피처 생성
- 인지 프로파일별 분석 및 모델링

## 검사 설명 요약
- **A 검사 (신규 자격 검사)**: 지각운동요인, 지적운동요인, 정서/행동 안정성
- **B 검사 (자격 유지 검사)**: 시야각, 시각기억, 주의지속, 다중과제 등 실전 운전 능력

## 진행 순서
1. 데이터 로딩 및 구조 파악
2. ID 기준 정리 (PrimaryKey 기준)
3. A 검사별 피처 생성 (A1~A9)
4. B 검사별 피처 생성 (B1~B10)
5. 인지 프로파일 그룹화
6. EDA 및 상관관계 분석
7. 모델링 준비


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from typing import Dict, List, Tuple
import warnings
warnings.filterwarnings('ignore')

# 병렬 처리
from joblib import Parallel, delayed
import multiprocessing

# 머신러닝 라이브러리
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, classification_report
from sklearn.calibration import calibration_curve

# LightGBM (선택적)
try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except ImportError:
    HAS_LIGHTGBM = False
    print("LightGBM not available. Install with: pip install lightgbm")
    print("  -> Will use HistGradientBoostingClassifier instead")

# 차원 축소 (UMAP은 선택적)
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("UMAP not available. Install with: pip install umap-learn")

# SHAP (선택적)
try:
    import shap
    HAS_SHAP = True
except ImportError:
    HAS_SHAP = False
    print("SHAP not available. Install with: pip install shap")

# 시각화 설정
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 10
sns.set_style('whitegrid')

# 한글 폰트 설정 (필요시)
# plt.rcParams['font.family'] = 'DejaVu Sans'

print("\n라이브러리 로드 완료!")
print(f"  - LightGBM available: {HAS_LIGHTGBM}")
print(f"  - UMAP available: {HAS_UMAP}")
print(f"  - SHAP available: {HAS_SHAP}")

if not HAS_LIGHTGBM:
    print("\n⚠️  LightGBM이 없습니다. 피처 중요도 분석과 모델링 섹션에서")
    print("   HistGradientBoostingClassifier를 사용합니다.")
    print("   LightGBM을 설치하려면: pip install lightgbm")


UMAP not available. Install with: pip install umap-learn
SHAP not available. Install with: pip install shap

라이브러리 로드 완료!
  - LightGBM available: True
  - UMAP available: False
  - SHAP available: False


In [2]:
# =============================================================================
# 유틸리티 함수 정의
# =============================================================================

# =============================================================================
# CSV 파싱 함수 (best08.py와 동일)
# =============================================================================

def seq_mean(series: pd.Series) -> pd.Series:
    """CSV 형식 시리즈의 평균 계산 (best08.py와 동일)"""
    s = series.fillna("").astype(str)
    df = s.str.split(",", expand=True).replace("", np.nan).astype(float)
    arr = df.to_numpy()
    with np.errstate(invalid="ignore"):
        m = np.nanmean(arr, axis=1)
    return pd.Series(m, index=series.index)

def seq_std(series: pd.Series) -> pd.Series:
    """CSV 형식 시리즈의 표준편차 계산 (best08.py와 동일)"""
    s = series.fillna("").astype(str)
    df = s.str.split(",", expand=True).replace("", np.nan).astype(float)
    arr = df.to_numpy()
    with np.errstate(invalid="ignore"):
        std = np.nanstd(arr, axis=1)
    return pd.Series(std, index=series.index)

def seq_rate(series: pd.Series, target: str = "1") -> pd.Series:
    """CSV 형식 시리즈에서 target 값의 비율 계산 (best08.py와 동일)"""
    s = series.fillna("").astype(str)
    df = s.str.split(",", expand=True)
    arr = df.to_numpy()
    non_empty = (arr != "")
    denom = non_empty.sum(axis=1)
    mask = (arr == str(target))
    num = mask.sum(axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        rate = num / np.where(denom == 0, np.nan, denom)
    return pd.Series(rate, index=series.index)

def masked_mean_from_csv_series(cond_series, val_series, mask_val):
    """조건별로 마스킹하여 값의 평균 계산 (best08.py와 동일)"""
    cond_df = cond_series.fillna("").str.split(",", expand=True).replace("", np.nan)
    val_df  = val_series.fillna("").str.split(",", expand=True).replace("", np.nan)
    cond_arr = cond_df.to_numpy(dtype=float)
    val_arr  = val_df.to_numpy(dtype=float)
    mask = (cond_arr == mask_val)
    with np.errstate(invalid="ignore"):
        sums   = np.nansum(np.where(mask, val_arr, np.nan), axis=1)
        counts = np.sum(mask, axis=1)
        out = sums / np.where(counts == 0, np.nan, counts)
    return pd.Series(out, index=cond_series.index)

def parse_csv_series_to_array(series: pd.Series) -> np.ndarray:
    """
    CSV 형식으로 저장된 시리즈를 numpy 배열로 변환
    각 행을 리스트로 변환 (가변 길이)
    """
    def _parse(x):
        if pd.isna(x) or x == '':
            return []
        if isinstance(x, (int, float)):
            return [float(x)]
        try:
            s = str(x).strip()
            if s == '':
                return []
            return [float(v) for v in s.split(',') if v.strip() != '']
        except:
            return []
    
    return series.apply(_parse).values

def calculate_statistics(values: pd.Series, prefix: str = '') -> Dict[str, float]:
    """
    값들의 통계량 계산 (mean, median, std, min, max, q25, q75)
    """
    if len(values) == 0 or values.isna().all():
        return {}
    
    values_clean = values.dropna()
    if len(values_clean) == 0:
        return {}
    
    stats_dict = {
        f'{prefix}mean': values_clean.mean(),
        f'{prefix}median': values_clean.median(),
        f'{prefix}std': values_clean.std(),
        f'{prefix}min': values_clean.min(),
        f'{prefix}max': values_clean.max(),
        f'{prefix}q25': values_clean.quantile(0.25),
        f'{prefix}q75': values_clean.quantile(0.75),
    }
    return stats_dict

print("유틸리티 함수 정의 완료! (best08.py와 호환)")

def parse_csv_series(series: pd.Series) -> pd.DataFrame:
    """CSV 형식 시리즈를 DataFrame으로 변환 (각 값을 컬럼으로)"""
    if len(series) == 0:
        return pd.DataFrame()
    
    def _parse(x):
        if pd.isna(x) or x == '':
            return []
        if isinstance(x, (int, float)):
            return [float(x)]
        try:
            s = str(x).strip()
            if s == '':
                return []
            return [float(v) for v in s.split(',') if v.strip() != '']
        except:
            return []
    
    # 각 행을 파싱하여 최대 길이 확인
    parsed = series.apply(_parse)
    max_len = parsed.apply(len).max() if len(parsed) > 0 else 0
    
    if max_len == 0:
        return pd.DataFrame()
    
    # DataFrame으로 변환
    data = {}
    for i in range(max_len):
        data[i] = parsed.apply(lambda x: x[i] if i < len(x) else np.nan)
    
    return pd.DataFrame(data)

print('유틸리티 함수 정의 완료!')

유틸리티 함수 정의 완료! (best08.py와 호환)
유틸리티 함수 정의 완료!


### 6-2. A1 검사 피처 생성 (속도 예측 - 터널 통과)

**검사 설명**: 빠른/중간/느린 속도로 이동하는 차량이 터널을 통과할 때 목표지점 도착을 예측

**피처 전략**:
- 속도별 정확도 (slow/normal/fast)
- 속도별 반응시간 통계
- 예측 편향 (편차거리)
- 좌/우 조건 민감도


In [3]:
def extract_A1_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A1 검사 (속도 예측) 피처 추출 (best08.py 방식 활용)
    
    컬럼:
    - A1-1: Condition 1 (1=left, 2=right) - CSV 형식
    - A1-2: Condition 2 (1=slow, 2=normal, 3=fast) - CSV 형식
    - A1-3: Response (0=N, 1=Y) - CSV 형식
    - A1-4: ResponseTime - CSV 형식
    """
    features = {}
    
    if 'A1-3' not in df.columns or 'A1-4' not in df.columns:
        return features
    
    # best08.py 방식 사용
    if len(df) == 0:
        return features
    
    row = df.iloc[0]
    
    # 전체 정확도 (Response rate)
    try:
        resp_rate = seq_rate(pd.Series([row['A1-3']]), "1").iloc[0]
        if pd.notna(resp_rate):
            features['A1_acc_overall'] = float(resp_rate)
    except:
        pass
    
    # RT 통계
    try:
        rt_mean = seq_mean(pd.Series([row['A1-4']])).iloc[0]
        rt_std = seq_std(pd.Series([row['A1-4']])).iloc[0]
        if pd.notna(rt_mean):
            features['A1_rt_mean'] = float(rt_mean)
        if pd.notna(rt_std):
            features['A1_rt_std'] = float(rt_std)
    except:
        pass
    
    # 좌/우 조건별 RT
    if 'A1-1' in df.columns:
        try:
            rt_left = masked_mean_from_csv_series(
                pd.Series([row['A1-1']]), 
                pd.Series([row['A1-4']]), 
                1
            ).iloc[0]
            rt_right = masked_mean_from_csv_series(
                pd.Series([row['A1-1']]), 
                pd.Series([row['A1-4']]), 
                2
            ).iloc[0]
            if pd.notna(rt_left) and pd.notna(rt_right):
                features['A1_rt_left'] = float(rt_left)
                features['A1_rt_right'] = float(rt_right)
                features['A1_rt_side_diff'] = float(rt_left - rt_right)
        except:
            pass
    
    # 속도별 RT (slow/normal/fast)
    if 'A1-2' in df.columns:
        try:
            rt_slow = masked_mean_from_csv_series(
                pd.Series([row['A1-2']]), 
                pd.Series([row['A1-4']]), 
                1
            ).iloc[0]
            rt_fast = masked_mean_from_csv_series(
                pd.Series([row['A1-2']]), 
                pd.Series([row['A1-4']]), 
                3
            ).iloc[0]
            if pd.notna(rt_slow) and pd.notna(rt_fast):
                features['A1_rt_slow'] = float(rt_slow)
                features['A1_rt_fast'] = float(rt_fast)
                features['A1_rt_speed_diff'] = float(rt_slow - rt_fast)
            
            # 속도별 정확도
            for speed_idx, speed_name in [(1, 'slow'), (2, 'normal'), (3, 'fast')]:
                try:
                    # 해당 속도의 Response 비율
                    speed_resp_series = pd.Series([row['A1-3']])
                    speed_cond_series = pd.Series([row['A1-2']])
                    # 조건에 맞는 응답만 필터링하여 정확도 계산
                    # CSV를 파싱하여 조건별로 분리
                    cond_arr = parse_csv_series_to_array(speed_cond_series)[0]
                    resp_arr = parse_csv_series_to_array(speed_resp_series)[0]
                    
                    if len(cond_arr) == len(resp_arr) and len(cond_arr) > 0:
                        speed_mask = np.array(cond_arr) == speed_idx
                        speed_resp = np.array(resp_arr)[speed_mask]
                        if len(speed_resp) > 0:
                            features[f'A1_acc_{speed_name}'] = float((speed_resp == 1).mean())
                except:
                    pass
            
            # 속도별 정확도 차이
            if 'A1_acc_slow' in features and 'A1_acc_fast' in features:
                features['A1_acc_slow_fast_diff'] = features['A1_acc_slow'] - features['A1_acc_fast']
        except:
            pass
    
    return features

print("A1 피처 추출 함수 정의 완료! (best08.py 방식 적용)")


A1 피처 추출 함수 정의 완료! (best08.py 방식 적용)


### 6-6. A2 검사 피처 생성 (브레이크 페달 - 정지거리 예측)

**검사 설명**: 이동선이 정지선에 멈출 수 있도록 예측하고 브레이크 페달 밟기

**피처 전략**:
- 속도·가속도 조건별 정답률 / 편차 평균, 편차 절댓값 평균
- 브레이크 타이밍 일관성 (RT 표준편차, IQR)
- 과도/과소 제동 패턴 (편차의 부호 비율)


In [4]:
def extract_A2_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A2 검사 (브레이크 페달 - 정지거리 예측) 피처 추출
    
    컬럼:
    - A2-1: Condition 1 (1=slow, 2=normal, 3=fast)
    - A2-2: Condition 2 (1=slow, 2=normal, 3=fast)
    - A2-3: Response (0=N, 1=Y)
    - A2-4: ResponseTime
    """
    features = {}
    
    if 'A2-3' not in df.columns or 'A2-4' not in df.columns:
        return features
    
    # 데이터 파싱
    responses = parse_csv_series(df['A2-3']) if df['A2-3'].dtype == 'object' else df[['A2-3']]
    rt_series = parse_csv_series(df['A2-4']) if df['A2-4'].dtype == 'object' else df[['A2-4']]
    
    if isinstance(df['A2-3'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['A2-3']]
    if isinstance(df['A2-4'].iloc[0] if len(df) > 0 else None, (int, float)):
        rt_series = df[['A2-4']]
    
    # 전체 정확도 & RT
    if len(responses) > 0:
        all_responses = responses.values.flatten()
        all_responses = all_responses[~np.isnan(all_responses)]
        if len(all_responses) > 0:
            features['A2_acc_overall'] = (all_responses == 1).mean()
            features['A2_total_trials'] = len(all_responses)
    
    if len(rt_series) > 0:
        all_rt = rt_series.values.flatten()
        all_rt = all_rt[~np.isnan(all_rt)]
        if len(all_rt) > 0:
            rt_stats = calculate_statistics(pd.Series(all_rt), prefix='A2_rt_')
            features.update(rt_stats)
            # 브레이크 타이밍 일관성
            features['A2_rt_std'] = np.std(all_rt)
            features['A2_rt_iqr'] = np.percentile(all_rt, 75) - np.percentile(all_rt, 25)
            features['A2_rt_cv'] = np.std(all_rt) / (np.mean(all_rt) + 1e-6)  # 변동계수
    
    # Condition별 분석 (속도별)
    if 'A2-1' in df.columns:
        condition1 = parse_csv_series(df['A2-1']) if df['A2-1'].dtype == 'object' else df[['A2-1']]
        if isinstance(df['A2-1'].iloc[0] if len(df) > 0 else None, (int, float)):
            condition1 = df[['A2-1']]
        
        if len(condition1) > 0 and len(responses) > 0:
            for speed_idx, speed_name in [(1, 'slow'), (2, 'normal'), (3, 'fast')]:
                speed_mask = condition1.values.flatten() == speed_idx
                speed_responses = responses.values.flatten()[speed_mask]
                speed_responses = speed_responses[~np.isnan(speed_responses)]
                speed_rt = rt_series.values.flatten()[speed_mask]
                speed_rt = speed_rt[~np.isnan(speed_rt)]
                
                if len(speed_responses) > 0:
                    features[f'A2_acc_{speed_name}'] = (speed_responses == 1).mean()
                if len(speed_rt) > 0:
                    features[f'A2_rt_{speed_name}_mean'] = np.mean(speed_rt)
                    features[f'A2_rt_{speed_name}_std'] = np.std(speed_rt)
            
            # 속도별 정확도 차이
            if 'A2_acc_slow' in features and 'A2_acc_fast' in features:
                features['A2_acc_slow_fast_diff'] = features['A2_acc_slow'] - features['A2_acc_fast']
    
    return features

print("A2 피처 추출 함수 정의 완료!")


A2 피처 추출 함수 정의 완료!


### 6-7. A3 검사 피처 생성 (주의 전환 - valid/invalid cue)

**검사 설명**: 우물정자(#) 위치에 화살표가 나타날 확률 75%, 주의 전환 능력 측정

**피처 전략**:
- Valid vs Invalid 정확도 & RT
- 주의 전환 비용 (rt_invalid - rt_valid, acc_valid - acc_invalid)
- 원 크기별 성능 (small vs big)
- 좌/우 비대칭


In [5]:
def extract_A3_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A3 검사 (주의 전환) 피처 추출
    
    컬럼:
    - A3-1: Condition 1 (1=small, 2=big)
    - A3-2: Condition 2 (1-8 clockwise)
    - A3-3: Condition 3 (1=left, 2=right)
    - A3-4: Condition 4 (1-8 clockwise)
    - A3-5: Response 1 (1=valid correct, 2=valid incorrect, 3=invalid correct, 4=invalid incorrect)
    - A3-6: Response 2 (0=N, 1=Y)
    - A3-7: ResponseTime
    """
    features = {}
    
    if 'A3-5' not in df.columns or 'A3-7' not in df.columns:
        return features
    
    # 데이터 파싱
    responses = parse_csv_series(df['A3-5']) if df['A3-5'].dtype == 'object' else df[['A3-5']]
    rt_series = parse_csv_series(df['A3-7']) if df['A3-7'].dtype == 'object' else df[['A3-7']]
    
    if isinstance(df['A3-5'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['A3-5']]
    if isinstance(df['A3-7'].iloc[0] if len(df) > 0 else None, (int, float)):
        rt_series = df[['A3-7']]
    
    if len(responses) == 0 or len(rt_series) == 0:
        return features
    
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    all_rt = rt_series.values.flatten()
    all_rt = all_rt[~np.isnan(all_rt)]
    
    if len(all_responses) == 0:
        return features
    
    # Valid vs Invalid 분석
    # Response 1: 1=valid correct, 2=valid incorrect, 3=invalid correct, 4=invalid incorrect
    valid_mask = (all_responses == 1) | (all_responses == 2)  # valid
    invalid_mask = (all_responses == 3) | (all_responses == 4)  # invalid
    
    valid_correct = (all_responses == 1).sum()
    valid_total = valid_mask.sum()
    invalid_correct = (all_responses == 3).sum()
    invalid_total = invalid_mask.sum()
    
    if valid_total > 0:
        features['A3_acc_valid'] = valid_correct / valid_total
    if invalid_total > 0:
        features['A3_acc_invalid'] = invalid_correct / invalid_total
    
    # Valid/Invalid RT
    if valid_mask.sum() > 0 and len(all_rt) >= len(all_responses):
        valid_rt = all_rt[valid_mask[:len(all_rt)]]
        valid_rt = valid_rt[~np.isnan(valid_rt)]
        if len(valid_rt) > 0:
            features['A3_rt_valid_mean'] = np.mean(valid_rt)
            features['A3_rt_valid_std'] = np.std(valid_rt)
    
    if invalid_mask.sum() > 0 and len(all_rt) >= len(all_responses):
        invalid_rt = all_rt[invalid_mask[:len(all_rt)]]
        invalid_rt = invalid_rt[~np.isnan(invalid_rt)]
        if len(invalid_rt) > 0:
            features['A3_rt_invalid_mean'] = np.mean(invalid_rt)
            features['A3_rt_invalid_std'] = np.std(invalid_rt)
    
    # 주의 전환 비용 (Attention Switching Cost)
    if 'A3_rt_valid_mean' in features and 'A3_rt_invalid_mean' in features:
        features['A3_switch_cost_rt'] = features['A3_rt_invalid_mean'] - features['A3_rt_valid_mean']
    if 'A3_acc_valid' in features and 'A3_acc_invalid' in features:
        features['A3_switch_cost_acc'] = features['A3_acc_valid'] - features['A3_acc_invalid']
    
    # 원 크기별 성능 (small vs big)
    if 'A3-1' in df.columns:
        condition1 = parse_csv_series(df['A3-1']) if df['A3-1'].dtype == 'object' else df[['A3-1']]
        if isinstance(df['A3-1'].iloc[0] if len(df) > 0 else None, (int, float)):
            condition1 = df[['A3-1']]
        
        if len(condition1) > 0:
            small_mask = condition1.values.flatten() == 1
            big_mask = condition1.values.flatten() == 2
            
            # Small circle 성능
            if small_mask.sum() > 0 and len(all_responses) >= len(condition1.values.flatten()):
                small_responses = all_responses[small_mask[:len(all_responses)]]
                small_correct = (small_responses == 1).sum()  # valid correct만
                small_total = len(small_responses)
                if small_total > 0:
                    features['A3_acc_small'] = small_correct / small_total
            
            # Big circle 성능 (peripheral)
            if big_mask.sum() > 0 and len(all_responses) >= len(condition1.values.flatten()):
                big_responses = all_responses[big_mask[:len(all_responses)]]
                big_correct = (big_responses == 1).sum()
                big_total = len(big_responses)
                if big_total > 0:
                    features['A3_acc_big'] = big_correct / big_total
            
            # Peripheral 성능 저하
            if 'A3_acc_small' in features and 'A3_acc_big' in features:
                features['A3_peripheral_deficit'] = features['A3_acc_small'] - features['A3_acc_big']
    
    # 좌/우 비대칭
    if 'A3-3' in df.columns:
        condition3 = parse_csv_series(df['A3-3']) if df['A3-3'].dtype == 'object' else df[['A3-3']]
        if isinstance(df['A3-3'].iloc[0] if len(df) > 0 else None, (int, float)):
            condition3 = df[['A3-3']]
        
        if len(condition3) > 0:
            left_mask = condition3.values.flatten() == 1
            right_mask = condition3.values.flatten() == 2
            
            if left_mask.sum() > 0 and len(all_responses) >= len(condition3.values.flatten()):
                left_responses = all_responses[left_mask[:len(all_responses)]]
                left_correct = (left_responses == 1).sum()
                left_total = len(left_responses)
                if left_total > 0:
                    features['A3_acc_left'] = left_correct / left_total
            
            if right_mask.sum() > 0 and len(all_responses) >= len(condition3.values.flatten()):
                right_responses = all_responses[right_mask[:len(all_responses)]]
                right_correct = (right_responses == 1).sum()
                right_total = len(right_responses)
                if right_total > 0:
                    features['A3_acc_right'] = right_correct / right_total
            
            # 좌/우 비대칭
            if 'A3_acc_left' in features and 'A3_acc_right' in features:
                features['A3_left_right_asymmetry'] = abs(features['A3_acc_left'] - features['A3_acc_right'])
    
    return features

print("A3 피처 추출 함수 정의 완료!")


A3 피처 추출 함수 정의 완료!


### 6-4. A4 검사 피처 생성 (Stroop - 선택적 주의)

**검사 설명**: 색에만 반응 - Congruent/Incongruent 조건

**피처 전략**:
- Congruent/Incongruent 정확도/RT
- Stroop 간섭 효과 (rt_incon - rt_con, acc_con - acc_incon)
- 3초 초과/미응답 비율

In [6]:
def extract_A4_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A4 검사 (Stroop - 선택적 주의) 피처 추출
    
    컬럼:
    - A4-1: Condition 1 (1=congruent, 2=incongruent)
    - A4-2: Condition 2 (1=red, 2=green)
    - A4-3: Response 1 (1=correct, 2=incorrect)
    - A4-4: Response 2 (0=N, 1=Y)
    - A4-5: ResponseTime
    """
    features = {}
    
    if 'A4-3' not in df.columns or 'A4-5' not in df.columns:
        return features
    
    # 데이터 파싱
    responses = parse_csv_series(df['A4-3']) if df['A4-3'].dtype == 'object' else df[['A4-3']]
    rt_series = parse_csv_series(df['A4-5']) if df['A4-5'].dtype == 'object' else df[['A4-5']]
    
    if isinstance(df['A4-3'].iloc[0], (int, float)):
        responses = df[['A4-3']]
    if isinstance(df['A4-5'].iloc[0], (int, float)):
        rt_series = df[['A4-5']]
    
    # 전체 정확도 & RT
    if len(responses) > 0:
        all_responses = responses.values.flatten()
        all_responses = all_responses[~np.isnan(all_responses)]
        if len(all_responses) > 0:
            features['A4_acc_overall'] = (all_responses == 1).mean()
            features['A4_total_trials'] = len(all_responses)
    
    if len(rt_series) > 0:
        all_rt = rt_series.values.flatten()
        all_rt = all_rt[~np.isnan(all_rt)]
        if len(all_rt) > 0:
            rt_stats = calculate_statistics(pd.Series(all_rt), prefix='A4_rt_')
            features.update(rt_stats)
            # 3초 초과 비율
            features['A4_rt_over3s_ratio'] = (all_rt > 3000).mean() if len(all_rt) > 0 else 0.0
    
    # Congruent vs Incongruent 분석
    if 'A4-1' in df.columns:
        condition1 = parse_csv_series(df['A4-1']) if df['A4-1'].dtype == 'object' else df[['A4-1']]
        if isinstance(df['A4-1'].iloc[0], (int, float)):
            condition1 = df[['A4-1']]
        
        if len(condition1) > 0 and len(responses) > 0:
            # Congruent (1)
            con_mask = condition1.values.flatten() == 1
            con_responses = responses.values.flatten()[con_mask]
            con_responses = con_responses[~np.isnan(con_responses)]
            con_rt = rt_series.values.flatten()[con_mask]
            con_rt = con_rt[~np.isnan(con_rt)]
            
            if len(con_responses) > 0:
                features['A4_acc_congruent'] = (con_responses == 1).mean()
            if len(con_rt) > 0:
                features['A4_rt_congruent_mean'] = np.mean(con_rt)
                features['A4_rt_congruent_std'] = np.std(con_rt)
            
            # Incongruent (2)
            incon_mask = condition1.values.flatten() == 2
            incon_responses = responses.values.flatten()[incon_mask]
            incon_responses = incon_responses[~np.isnan(incon_responses)]
            incon_rt = rt_series.values.flatten()[incon_mask]
            incon_rt = incon_rt[~np.isnan(incon_rt)]
            
            if len(incon_responses) > 0:
                features['A4_acc_incongruent'] = (incon_responses == 1).mean()
            if len(incon_rt) > 0:
                features['A4_rt_incongruent_mean'] = np.mean(incon_rt)
                features['A4_rt_incongruent_std'] = np.std(incon_rt)
            
            # Stroop 간섭 효과 (Interference Effect)
            if 'A4_rt_congruent_mean' in features and 'A4_rt_incongruent_mean' in features:
                features['A4_stroop_rt_effect'] = features['A4_rt_incongruent_mean'] - features['A4_rt_congruent_mean']
            if 'A4_acc_congruent' in features and 'A4_acc_incongruent' in features:
                features['A4_stroop_acc_effect'] = features['A4_acc_congruent'] - features['A4_acc_incongruent']
    
    return features

print("A4 피처 추출 함수 정의 완료!")


A4 피처 추출 함수 정의 완료!


### 6-8. A5 검사 피처 생성 (변화 탐지)

**검사 설명**: 색상/위치/모양 변화를 탐지하는 능력

**피처 전략**:
- 조건별 정확도 (non/pos/color/shape change)
- 오류 유형 (miss, false alarm)
- Hit rate, Correct rejection rate
- RT 평균/분산


In [7]:
def extract_A5_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A5 검사 (변화 탐지) 피처 추출
    
    컬럼:
    - A5-1: Condition (1=non change, 2=pos change, 3=color change, 4=shape change)
    - A5-2: Response 1 (1=correct answer, 2=incorrect answer)
    - A5-3: Response 2 (0=N, 1=Y)
    """
    features = {}
    
    if 'A5-1' not in df.columns or 'A5-2' not in df.columns:
        return features
    
    # 데이터 파싱
    conditions = parse_csv_series(df['A5-1']) if df['A5-1'].dtype == 'object' else df[['A5-1']]
    responses = parse_csv_series(df['A5-2']) if df['A5-2'].dtype == 'object' else df[['A5-2']]
    
    if isinstance(df['A5-1'].iloc[0] if len(df) > 0 else None, (int, float)):
        conditions = df[['A5-1']]
    if isinstance(df['A5-2'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['A5-2']]
    
    if len(conditions) == 0 or len(responses) == 0:
        return features
    
    all_conditions = conditions.values.flatten()
    all_conditions = all_conditions[~np.isnan(all_conditions)]
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    
    if len(all_conditions) == 0 or len(all_responses) == 0:
        return features
    
    # 조건별 정확도
    condition_names = {1: 'non', 2: 'pos', 3: 'color', 4: 'shape'}
    for cond_idx, cond_name in condition_names.items():
        cond_mask = all_conditions == cond_idx
        if cond_mask.sum() > 0:
            cond_responses = all_responses[cond_mask[:len(all_responses)]]
            correct = (cond_responses == 1).sum()
            total = len(cond_responses)
            if total > 0:
                features[f'A5_acc_{cond_name}'] = correct / total
    
    # 오류 유형 분석
    # Change인데 예라고 한 경우 (hit) - pos/color/shape change에서 correct
    change_mask = (all_conditions == 2) | (all_conditions == 3) | (all_conditions == 4)
    change_total = change_mask.sum()
    if change_total > 0:
        change_responses = all_responses[change_mask[:len(all_responses)]]
        hits = (change_responses == 1).sum()  # change인데 correct
        features['A5_hit_rate'] = hits / change_total if change_total > 0 else 0.0
    
    # Non-change인데 아니오라고 한 경우 (false alarm) - non change에서 incorrect
    non_change_mask = all_conditions == 1
    non_change_total = non_change_mask.sum()
    if non_change_total > 0:
        non_change_responses = all_responses[non_change_mask[:len(all_responses)]]
        false_alarms = (non_change_responses == 2).sum()  # non-change인데 incorrect
        features['A5_false_alarm_rate'] = false_alarms / non_change_total if non_change_total > 0 else 0.0
    
    # Correct rejection rate (non-change에서 correct)
    if non_change_total > 0:
        correct_rejections = (non_change_responses == 1).sum()
        features['A5_correct_rejection_rate'] = correct_rejections / non_change_total if non_change_total > 0 else 0.0
    
    # 변화 유형별 취약점
    if 'A5_acc_color' in features and 'A5_acc_shape' in features and 'A5_acc_pos' in features:
        # 어떤 변화에 가장 취약한지
        min_acc = min(features.get('A5_acc_color', 1.0), 
                     features.get('A5_acc_shape', 1.0),
                     features.get('A5_acc_pos', 1.0))
        features['A5_min_change_acc'] = min_acc
    
    return features

print("A5 피처 추출 함수 정의 완료!")


A5 피처 추출 함수 정의 완료!


### 6-9. A6/A7 검사 피처 생성 (문제풀이 - 지적운동요인)

**검사 설명**: 
- A6: 관계변화 추론 (14문항, 7분 제한)
- A7: 단순/복잡도형 찾기 (18문항, 10분 제한)

**피처 전략**:
- 정답 수, 오답 수, 미응답 수
- 초반/후반 난이도별 성능
- (가능하면) 각 문항별 소요시간


In [8]:
def extract_A6_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A6 검사 (관계변화 추론) 피처 추출
    
    컬럼:
    - A6-1: 개수 (정답 수로 추정)
    """
    features = {}
    
    if 'A6-1' in df.columns:
        value = df['A6-1'].iloc[0] if len(df) > 0 else None
        if pd.notna(value):
            features['A6_correct_count'] = float(value)
            # 전체 14문항 중 정답 수이므로 정확도 추정
            features['A6_acc'] = float(value) / 14.0 if value <= 14 else 1.0
            # 미응답 수 추정 (시간 초과)
            features['A6_missed'] = 14.0 - float(value) if value <= 14 else 0.0
    
    return features

def extract_A7_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A7 검사 (단순/복잡도형 찾기) 피처 추출
    
    컬럼:
    - A7-1: 개수 (정답 수로 추정)
    """
    features = {}
    
    if 'A7-1' in df.columns:
        value = df['A7-1'].iloc[0] if len(df) > 0 else None
        if pd.notna(value):
            features['A7_correct_count'] = float(value)
            # 전체 18문항 중 정답 수이므로 정확도 추정
            features['A7_acc'] = float(value) / 18.0 if value <= 18 else 1.0
            # 미응답 수 추정 (시간 초과)
            features['A7_missed'] = 18.0 - float(value) if value <= 18 else 0.0
    
    return features

print("A6, A7 피처 추출 함수 정의 완료!")


A6, A7 피처 추출 함수 정의 완료!


### 6-10. A8 검사 피처 생성 (타당도 척도)

**검사 설명**: 응답 왜곡 정도, 반응 일관성 정도

**피처 전략**:
- 응답 왜곡 점수
- 반응 일관성 점수
- 신뢰도 낮은 응답자 플래그


In [9]:
def extract_A8_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A8 검사 (타당도 척도) 피처 추출
    
    컬럼:
    - A8-1: 타당도 척도 1 (응답 왜곡 정도)
    - A8-2: 타당도 척도 2 (반응 일관성 정도)
    """
    features = {}
    
    if 'A8-1' in df.columns:
        value = df['A8-1'].iloc[0] if len(df) > 0 else None
        if pd.notna(value):
            features['A8_response_distortion'] = float(value)
            # 신뢰도 낮은 응답자 플래그 (임계값은 데이터에 따라 조정 필요)
            features['A8_high_distortion'] = 1.0 if float(value) > 50 else 0.0  # 임계값 예시
    
    if 'A8-2' in df.columns:
        value = df['A8-2'].iloc[0] if len(df) > 0 else None
        if pd.notna(value):
            features['A8_response_consistency'] = float(value)
            # 일관성 낮은 응답자 플래그
            features['A8_low_consistency'] = 1.0 if float(value) < 50 else 0.0  # 임계값 예시
    
    # 신뢰도 종합 지표
    if 'A8_response_distortion' in features and 'A8_response_consistency' in features:
        # 왜곡이 높고 일관성이 낮으면 신뢰도 낮음
        features['A8_low_reliability'] = 1.0 if (
            features.get('A8_high_distortion', 0) == 1.0 or 
            features.get('A8_low_consistency', 0) == 1.0
        ) else 0.0
    
    return features

print("A8 피처 추출 함수 정의 완료!")


A8 피처 추출 함수 정의 완료!


### 6-11. A9 검사 피처 생성 (질문지 - 정서/행동 안정성)

**검사 설명**: 질문지형 검사 - 정서안정성, 행동안정성, 현실판단력, 정신적민첩성, 생활스트레스

**피처 전략**:
- 각 척도별 점수
- 상호작용 피처 (stress - emotional_stability 등)
- 신뢰도 낮은 응답자 플래그

In [10]:
def extract_A9_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    A9 검사 (질문지 - 정서/행동 안정성) 피처 추출
    
    컬럼:
    - A9-1: 정서안정성
    - A9-2: 행동안정성
    - A9-3: 현실판단력
    - A9-4: 정신적민첩성
    - A9-5: 생활스트레스
    """
    features = {}
    
    a9_cols = ['A9-1', 'A9-2', 'A9-3', 'A9-4', 'A9-5']
    a9_names = ['emotional_stability', 'behavioral_stability', 'reality_judgment', 
                'mental_agility', 'life_stress']
    
    for col, name in zip(a9_cols, a9_names):
        if col in df.columns:
            value = df[col].iloc[0] if len(df) > 0 else None
            if pd.notna(value):
                features[f'A9_{name}'] = float(value)
    
    # 상호작용 피처
    if 'A9_life_stress' in features and 'A9_emotional_stability' in features:
        features['A9_stress_emotional_diff'] = features['A9_life_stress'] - features['A9_emotional_stability']
    
    if 'A9_mental_agility' in features and 'A9_behavioral_stability' in features:
        features['A9_agility_stability_sum'] = features['A9_mental_agility'] + features['A9_behavioral_stability']
    
    return features

print("A9 피처 추출 함수 정의 완료!")


A9 피처 추출 함수 정의 완료!


### 6-11. B1/B2 검사 피처 생성 (시야각 검사)

**검사 설명**: 주변시야에 보이는 자극을 정확하게 찾아내는 검사

**피처 전략**:
- Change vs Non-change 정확도
- Hit rate, False alarm rate
- Peripheral 자극일 때 정확도/RT


In [11]:
def extract_B1_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B1 검사 (시야각 검사) 피처 추출
    
    컬럼:
    - B1-1: Condition (1=color change, 2=color non change)
    - B1-2: 응답시간
    - B1-3: Response (1=change-correct, 2=change-incorrect, 3=non change-correct, 4=non change-incorrect)
    """
    features = {}
    
    if 'B1-3' not in df.columns:
        return features
    
    # 데이터 파싱
    responses = parse_csv_series(df['B1-3']) if df['B1-3'].dtype == 'object' else df[['B1-3']]
    
    if isinstance(df['B1-3'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['B1-3']]
    
    if len(responses) == 0:
        return features
    
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    
    if len(all_responses) == 0:
        return features
    
    # Change vs Non-change 분석
    # Response: 1=change-correct, 2=change-incorrect, 3=non change-correct, 4=non change-incorrect
    change_mask = (all_responses == 1) | (all_responses == 2)
    non_change_mask = (all_responses == 3) | (all_responses == 4)
    
    change_total = change_mask.sum()
    non_change_total = non_change_mask.sum()
    
    # Hit rate (change인데 correct)
    if change_total > 0:
        hits = (all_responses == 1).sum()
        features['B1_hit_rate'] = hits / change_total
    
    # False alarm rate (non-change인데 incorrect로 응답)
    if non_change_total > 0:
        false_alarms = (all_responses == 4).sum()  # non-change-incorrect
        features['B1_false_alarm_rate'] = false_alarms / non_change_total
    
    # Correct rejection rate
    if non_change_total > 0:
        correct_rejections = (all_responses == 3).sum()
        features['B1_correct_rejection_rate'] = correct_rejections / non_change_total
    
    # 전체 정확도
    total = len(all_responses)
    correct = ((all_responses == 1) | (all_responses == 3)).sum()
    if total > 0:
        features['B1_acc_overall'] = correct / total
    
    # RT 통계
    if 'B1-2' in df.columns:
        rt_series = parse_csv_series(df['B1-2']) if df['B1-2'].dtype == 'object' else df[['B1-2']]
        if isinstance(df['B1-2'].iloc[0] if len(df) > 0 else None, (int, float)):
            rt_series = df[['B1-2']]
        
        if len(rt_series) > 0:
            all_rt = rt_series.values.flatten()
            all_rt = all_rt[~np.isnan(all_rt)]
            if len(all_rt) > 0:
                rt_stats = calculate_statistics(pd.Series(all_rt), prefix='B1_rt_')
                features.update(rt_stats)
    
    return features

def extract_B2_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B2 검사 (시야각 검사) 피처 추출
    
    컬럼:
    - B2-1: Response (1=correct, 2=incorrect)
    - B2-2: 응답시간
    - B2-3: Response (1=change-correct, 2=change-incorrect, 3=non change-correct, 4=non change-incorrect)
    """
    features = {}
    
    if 'B2-3' not in df.columns:
        return features
    
    # B1과 동일한 구조
    responses = parse_csv_series(df['B2-3']) if df['B2-3'].dtype == 'object' else df[['B2-3']]
    
    if isinstance(df['B2-3'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['B2-3']]
    
    if len(responses) == 0:
        return features
    
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    
    if len(all_responses) == 0:
        return features
    
    change_mask = (all_responses == 1) | (all_responses == 2)
    non_change_mask = (all_responses == 3) | (all_responses == 4)
    
    change_total = change_mask.sum()
    non_change_total = non_change_mask.sum()
    
    if change_total > 0:
        hits = (all_responses == 1).sum()
        features['B2_hit_rate'] = hits / change_total
    
    if non_change_total > 0:
        false_alarms = (all_responses == 4).sum()
        features['B2_false_alarm_rate'] = false_alarms / non_change_total
        correct_rejections = (all_responses == 3).sum()
        features['B2_correct_rejection_rate'] = correct_rejections / non_change_total
    
    total = len(all_responses)
    correct = ((all_responses == 1) | (all_responses == 3)).sum()
    if total > 0:
        features['B2_acc_overall'] = correct / total
    
    # RT 통계
    if 'B2-2' in df.columns:
        rt_series = parse_csv_series(df['B2-2']) if df['B2-2'].dtype == 'object' else df[['B2-2']]
        if isinstance(df['B2-2'].iloc[0] if len(df) > 0 else None, (int, float)):
            rt_series = df[['B2-2']]
        
        if len(rt_series) > 0:
            all_rt = rt_series.values.flatten()
            all_rt = all_rt[~np.isnan(all_rt)]
            if len(all_rt) > 0:
                rt_stats = calculate_statistics(pd.Series(all_rt), prefix='B2_rt_')
                features.update(rt_stats)
    
    return features

print("B1, B2 피처 추출 함수 정의 완료!")


B1, B2 피처 추출 함수 정의 완료!


### 6-12. B3 검사 피처 생성 (반응속도)

**검사 설명**: 빨간 신호/위험표지 → 브레이크 페달까지의 RT

**피처 전략**:
- 평균 RT, median RT, 최소 RT, 최대 RT
- Miss count / incorrect count
- RT 표준편차 (일관성)


In [12]:
def extract_B3_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B3 검사 (반응속도) 피처 추출
    
    컬럼:
    - B3-1: Response (1=correct, 2=incorrect)
    - B3-2: 반응시간
    """
    features = {}
    
    if 'B3-1' not in df.columns or 'B3-2' not in df.columns:
        return features
    
    # 데이터 파싱
    responses = parse_csv_series(df['B3-1']) if df['B3-1'].dtype == 'object' else df[['B3-1']]
    rt_series = parse_csv_series(df['B3-2']) if df['B3-2'].dtype == 'object' else df[['B3-2']]
    
    if isinstance(df['B3-1'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['B3-1']]
    if isinstance(df['B3-2'].iloc[0] if len(df) > 0 else None, (int, float)):
        rt_series = df[['B3-2']]
    
    if len(responses) == 0 or len(rt_series) == 0:
        return features
    
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    all_rt = rt_series.values.flatten()
    all_rt = all_rt[~np.isnan(all_rt)]
    
    # 정확도
    total = len(all_responses)
    correct = (all_responses == 1).sum()
    incorrect = (all_responses == 2).sum()
    
    if total > 0:
        features['B3_acc'] = correct / total
        features['B3_miss_count'] = incorrect
        features['B3_miss_rate'] = incorrect / total
    
    # RT 통계
    if len(all_rt) > 0:
        rt_stats = calculate_statistics(pd.Series(all_rt), prefix='B3_rt_')
        features.update(rt_stats)
        # 일관성 지표
        features['B3_rt_std'] = np.std(all_rt)
        features['B3_rt_cv'] = np.std(all_rt) / (np.mean(all_rt) + 1e-6)
        
        # Correct trials만 RT 분석
        if len(all_rt) >= len(all_responses):
            correct_mask = all_responses == 1
            correct_rt = all_rt[correct_mask[:len(all_rt)]]
            correct_rt = correct_rt[~np.isnan(correct_rt)]
            if len(correct_rt) > 0:
                features['B3_rt_correct_mean'] = np.mean(correct_rt)
                features['B3_rt_correct_std'] = np.std(correct_rt)
    
    return features

print("B3 피처 추출 함수 정의 완료!")


B3 피처 추출 함수 정의 완료!


### 6-13. B4 검사 피처 생성 (선택적 주의력 - Flanker)

**검사 설명**: 화살표 방향 선택 - Congruent/Incongruent 조건

**피처 전략**:
- Congruent/Incongruent 정확도/RT
- 간섭효과 (rt_incon - rt_con, acc_con - acc_incon)
- 좌/우 방향 비대칭


In [13]:
def extract_B4_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B4 검사 (선택적 주의력 - Flanker) 피처 추출
    
    컬럼:
    - B4-1: Response (1=correct in congruent, 2=incorrect in congruent, 
                      3=correct in incongruent, 4=incorrect in incongruent)
    - B4-2: 반응시간
    """
    features = {}
    
    if 'B4-1' not in df.columns or 'B4-2' not in df.columns:
        return features
    
    # 데이터 파싱
    responses = parse_csv_series(df['B4-1']) if df['B4-1'].dtype == 'object' else df[['B4-1']]
    rt_series = parse_csv_series(df['B4-2']) if df['B4-2'].dtype == 'object' else df[['B4-2']]
    
    if isinstance(df['B4-1'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['B4-1']]
    if isinstance(df['B4-2'].iloc[0] if len(df) > 0 else None, (int, float)):
        rt_series = df[['B4-2']]
    
    if len(responses) == 0 or len(rt_series) == 0:
        return features
    
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    all_rt = rt_series.values.flatten()
    all_rt = all_rt[~np.isnan(all_rt)]
    
    # Congruent vs Incongruent 분석
    # Response: 1=con-correct, 2=con-incorrect, 3=incon-correct, 4=incon-incorrect
    con_mask = (all_responses == 1) | (all_responses == 2)
    incon_mask = (all_responses == 3) | (all_responses == 4)
    
    con_total = con_mask.sum()
    incon_total = incon_mask.sum()
    
    # Congruent 정확도
    if con_total > 0:
        con_correct = (all_responses == 1).sum()
        features['B4_acc_congruent'] = con_correct / con_total
    
    # Incongruent 정확도
    if incon_total > 0:
        incon_correct = (all_responses == 3).sum()
        features['B4_acc_incongruent'] = incon_correct / incon_total
    
    # RT 분석
    if len(all_rt) >= len(all_responses):
        con_rt = all_rt[con_mask[:len(all_rt)]]
        con_rt = con_rt[~np.isnan(con_rt)]
        incon_rt = all_rt[incon_mask[:len(all_rt)]]
        incon_rt = incon_rt[~np.isnan(incon_rt)]
        
        if len(con_rt) > 0:
            features['B4_rt_congruent_mean'] = np.mean(con_rt)
            features['B4_rt_congruent_std'] = np.std(con_rt)
        
        if len(incon_rt) > 0:
            features['B4_rt_incongruent_mean'] = np.mean(incon_rt)
            features['B4_rt_incongruent_std'] = np.std(incon_rt)
    
    # 간섭효과 (Interference Effect)
    if 'B4_rt_congruent_mean' in features and 'B4_rt_incongruent_mean' in features:
        features['B4_flanker_rt_effect'] = features['B4_rt_incongruent_mean'] - features['B4_rt_congruent_mean']
    if 'B4_acc_congruent' in features and 'B4_acc_incongruent' in features:
        features['B4_flanker_acc_effect'] = features['B4_acc_congruent'] - features['B4_acc_incongruent']
    
    # 전체 정확도
    total = len(all_responses)
    correct = ((all_responses == 1) | (all_responses == 3)).sum()
    if total > 0:
        features['B4_acc_overall'] = correct / total
    
    return features

print("B4 피처 추출 함수 정의 완료!")


B4 피처 추출 함수 정의 완료!


### 6-14. B5, B6, B7, B8 검사 피처 생성

**검사 설명**:
- B5: 공간 판단력 (얽힌 도로에서 경로 찾기)
- B6/B7: 시각적 기억 (교통안전표지/도로표지)
- B8: 주의 지속능력 (탑승차량 찾기)


In [14]:
def extract_B5_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B5 검사 (공간 판단력) 피처 추출
    
    컬럼:
    - B5-1: Response (1=correct, 2=incorrect)
    - B5-2: 반응시간
    """
    features = {}
    
    if 'B5-1' not in df.columns:
        return features
    
    responses = parse_csv_series(df['B5-1']) if df['B5-1'].dtype == 'object' else df[['B5-1']]
    if isinstance(df['B5-1'].iloc[0] if len(df) > 0 else None, (int, float)):
        responses = df[['B5-1']]
    
    if len(responses) == 0:
        return features
    
    all_responses = responses.values.flatten()
    all_responses = all_responses[~np.isnan(all_responses)]
    
    total = len(all_responses)
    correct = (all_responses == 1).sum()
    incorrect = (all_responses == 2).sum()
    
    if total > 0:
        features['B5_acc'] = correct / total
        features['B5_correct_count'] = correct
        features['B5_incorrect_count'] = incorrect
        
        # 앞/뒤 절반 성능 (피로도 측정)
        if total >= 2:
            half = total // 2
            first_half = all_responses[:half]
            second_half = all_responses[half:]
            features['B5_acc_first_half'] = (first_half == 1).mean()
            features['B5_acc_second_half'] = (second_half == 1).mean()
            features['B5_fatigue_effect'] = features['B5_acc_first_half'] - features['B5_acc_second_half']
    
    # RT 통계
    if 'B5-2' in df.columns:
        rt_series = parse_csv_series(df['B5-2']) if df['B5-2'].dtype == 'object' else df[['B5-2']]
        if isinstance(df['B5-2'].iloc[0] if len(df) > 0 else None, (int, float)):
            rt_series = df[['B5-2']]
        
        if len(rt_series) > 0:
            all_rt = rt_series.values.flatten()
            all_rt = all_rt[~np.isnan(all_rt)]
            if len(all_rt) > 0:
                rt_stats = calculate_statistics(pd.Series(all_rt), prefix='B5_rt_')
                features.update(rt_stats)
    
    return features

def extract_B6_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B6 검사 (시각적 기억 - 교통안전표지) 피처 추출
    
    컬럼:
    - B6: Response (1=correct, 2=incorrect)
    """
    features = {}
    
    if 'B6' in df.columns:
        responses = parse_csv_series(df['B6']) if df['B6'].dtype == 'object' else df[['B6']]
        if isinstance(df['B6'].iloc[0] if len(df) > 0 else None, (int, float)):
            responses = df[['B6']]
        
        if len(responses) > 0:
            all_responses = responses.values.flatten()
            all_responses = all_responses[~np.isnan(all_responses)]
            
            total = len(all_responses)
            correct = (all_responses == 1).sum()
            if total > 0:
                features['B6_acc'] = correct / total
                features['B6_correct_count'] = correct
    
    return features

def extract_B7_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B7 검사 (시각적 기억 - 도로표지) 피처 추출
    
    컬럼:
    - B7: Response (1=correct, 2=incorrect)
    """
    features = {}
    
    if 'B7' in df.columns:
        responses = parse_csv_series(df['B7']) if df['B7'].dtype == 'object' else df[['B7']]
        if isinstance(df['B7'].iloc[0] if len(df) > 0 else None, (int, float)):
            responses = df[['B7']]
        
        if len(responses) > 0:
            all_responses = responses.values.flatten()
            all_responses = all_responses[~np.isnan(all_responses)]
            
            total = len(all_responses)
            correct = (all_responses == 1).sum()
            if total > 0:
                features['B7_acc'] = correct / total
                features['B7_correct_count'] = correct
    
    # B6와 B7 통합 지표
    if 'B6_acc' in features and 'B7_acc' in features:
        features['B6_B7_acc_mean'] = (features['B6_acc'] + features['B7_acc']) / 2.0
        features['B6_B7_acc_diff'] = abs(features['B6_acc'] - features['B7_acc'])
    
    return features

def extract_B8_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B8 검사 (주의 지속능력) 피처 추출
    
    컬럼:
    - B8: Response (1=correct, 2=incorrect)
    """
    features = {}
    
    if 'B8' in df.columns:
        responses = parse_csv_series(df['B8']) if df['B8'].dtype == 'object' else df[['B8']]
        if isinstance(df['B8'].iloc[0] if len(df) > 0 else None, (int, float)):
            responses = df[['B8']]
        
        if len(responses) > 0:
            all_responses = responses.values.flatten()
            all_responses = all_responses[~np.isnan(all_responses)]
            
            total = len(all_responses)
            correct = (all_responses == 1).sum()
            if total > 0:
                features['B8_acc'] = correct / total
                features['B8_correct_count'] = correct
                
                # 난이도별 성능 (4조건이므로 12 trials / 4 = 3 trials per condition)
                # 초반/중반/후반 성능으로 난이도 변화 측정
                if total >= 3:
                    third = total // 3
                    first_third = all_responses[:third]
                    last_third = all_responses[-third:]
                    features['B8_acc_first_third'] = (first_third == 1).mean()
                    features['B8_acc_last_third'] = (last_third == 1).mean()
                    features['B8_sustained_attention_deficit'] = features['B8_acc_first_third'] - features['B8_acc_last_third']
    
    return features

print("B5, B6, B7, B8 피처 추출 함수 정의 완료!")


B5, B6, B7, B8 피처 추출 함수 정의 완료!


### 6-5. B9/B10 검사 피처 생성 (다중과제 - 시각+청각)

**검사 설명**: 장애물 회피 + 숫자 70 듣기 + 색상 반응 동시 수행

**피처 전략**:
- Auditory 성능 (hit_rate, fa_rate)
- Visual 성능 (장애물 회피, 색상 반응)
- 이중/삼중 과제 trade-off
- 시간 흐름에 따른 성능 변화


In [15]:
def extract_B9_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B9 검사 (다중과제) 피처 추출
    
    컬럼:
    - B9-1: aud, hit
    - B9-2: aud, miss
    - B9-3: aud, fa (false alarm)
    - B9-4: aud, cr (correct rejection)
    - B9-5: vis, err
    """
    features = {}
    
    b9_cols = ['B9-1', 'B9-2', 'B9-3', 'B9-4', 'B9-5']
    b9_names = ['aud_hit', 'aud_miss', 'aud_fa', 'aud_cr', 'vis_err']
    
    values = {}
    for col, name in zip(b9_cols, b9_names):
        if col in df.columns:
            value = df[col].iloc[0] if len(df) > 0 else 0
            if pd.notna(value):
                values[name] = float(value)
                features[f'B9_{name}'] = float(value)
    
    # Auditory 성능 지표
    if 'aud_hit' in values and 'aud_miss' in values:
        target_total = values['aud_hit'] + values['aud_miss']
        if target_total > 0:
            features['B9_aud_hit_rate'] = values['aud_hit'] / target_total
    
    if 'aud_fa' in values and 'aud_cr' in values:
        distractor_total = values['aud_fa'] + values['aud_cr']
        if distractor_total > 0:
            features['B9_aud_fa_rate'] = values['aud_fa'] / distractor_total
    
    # Visual 성능
    if 'vis_err' in values:
        features['B9_vis_error'] = values['vis_err']
    
    # 다중과제 trade-off (가능하면)
    if 'B9_aud_hit_rate' in features and 'B9_vis_error' in values:
        # hit rate가 높을수록 visual error가 높을 수 있음 (trade-off)
        pass
    
    return features

def extract_B10_features(df: pd.DataFrame) -> Dict[str, float]:
    """
    B10 검사 (다중과제) 피처 추출
    
    컬럼:
    - B10-1: aud, hit
    - B10-2: aud, miss
    - B10-3: aud, fa
    - B10-4: aud, cr
    - B10-5: vis1, err (장애물 회피)
    - B10-6: vis2, right response (색상 반응)
    """
    features = {}
    
    b10_cols = ['B10-1', 'B10-2', 'B10-3', 'B10-4', 'B10-5', 'B10-6']
    b10_names = ['aud_hit', 'aud_miss', 'aud_fa', 'aud_cr', 'vis1_err', 'vis2_correct']
    
    values = {}
    for col, name in zip(b10_cols, b10_names):
        if col in df.columns:
            value = df[col].iloc[0] if len(df) > 0 else 0
            if pd.notna(value):
                values[name] = float(value)
                features[f'B10_{name}'] = float(value)
    
    # Auditory 성능
    if 'aud_hit' in values and 'aud_miss' in values:
        target_total = values['aud_hit'] + values['aud_miss']
        if target_total > 0:
            features['B10_aud_hit_rate'] = values['aud_hit'] / target_total
    
    if 'aud_fa' in values and 'aud_cr' in values:
        distractor_total = values['aud_fa'] + values['aud_cr']
        if distractor_total > 0:
            features['B10_aud_fa_rate'] = values['aud_fa'] / distractor_total
    
    # Visual 성능
    if 'vis1_err' in values:
        features['B10_vis1_error'] = values['vis1_err']
    if 'vis2_correct' in values:
        features['B10_vis2_correct'] = values['vis2_correct']
    
    return features

print("B9, B10 피처 추출 함수 정의 완료!")


B9, B10 피처 추출 함수 정의 완료!


In [16]:
def extract_all_features_for_test_id(test_id: str, df_A: pd.DataFrame, df_B: pd.DataFrame) -> Dict[str, float]:
    """
    특정 Test_id에 대한 모든 피처 추출 (모든 검사 포함)
    
    Args:
        test_id: Test_id
        df_A: A 검사 데이터 (해당 Test_id만 포함)
        df_B: B 검사 데이터 (해당 Test_id만 포함)
    
    Returns:
        피처 딕셔너리
    """
    features = {}
    
    # Test_id 추가
    features['Test_id'] = test_id
    
    # A 검사 피처 추출
    if len(df_A) > 0:
        a_row = df_A.iloc[0]  # Test_id는 하나당 하나의 행
        a_df = pd.DataFrame([a_row])
        
        # A1: 속도 예측 (터널 통과)
        a1_feat = extract_A1_features(a_df)
        features.update(a1_feat)
        
        # A2: 브레이크 페달 (정지거리 예측)
        a2_feat = extract_A2_features(a_df)
        features.update(a2_feat)
        
        # A3: 주의 전환 (valid/invalid cue)
        a3_feat = extract_A3_features(a_df)
        features.update(a3_feat)
        
        # A4: Stroop (색에만 반응)
        a4_feat = extract_A4_features(a_df)
        features.update(a4_feat)
        
        # A5: 변화 탐지 (색/위치/모양)
        a5_feat = extract_A5_features(a_df)
        features.update(a5_feat)
        
        # A6: 관계변화 추론 (문제풀이)
        a6_feat = extract_A6_features(a_df)
        features.update(a6_feat)
        
        # A7: 단순/복잡도형 찾기 (문제풀이)
        a7_feat = extract_A7_features(a_df)
        features.update(a7_feat)
        
        # A8: 타당도 척도 (응답 왜곡, 일관성)
        a8_feat = extract_A8_features(a_df)
        features.update(a8_feat)
        
        # A9: 질문지 (정서/행동 안정성, 스트레스)
        a9_feat = extract_A9_features(a_df)
        features.update(a9_feat)
    
    # B 검사 피처 추출
    if len(df_B) > 0:
        b_row = df_B.iloc[0]  # Test_id는 하나당 하나의 행
        b_df = pd.DataFrame([b_row])
        
        # B1: 시야각 검사
        b1_feat = extract_B1_features(b_df)
        features.update(b1_feat)
        
        # B2: 시야각 검사
        b2_feat = extract_B2_features(b_df)
        features.update(b2_feat)
        
        # B3: 반응속도 (빨간 신호/위험표지 → 브레이크)
        b3_feat = extract_B3_features(b_df)
        features.update(b3_feat)
        
        # B4: 선택적 주의 (Flanker)
        b4_feat = extract_B4_features(b_df)
        features.update(b4_feat)
        
        # B5: 공간 판단력 (경로 찾기)
        b5_feat = extract_B5_features(b_df)
        features.update(b5_feat)
        
        # B6: 시각적 기억 (교통안전표지)
        b6_feat = extract_B6_features(b_df)
        features.update(b6_feat)
        
        # B7: 시각적 기억 (도로표지)
        b7_feat = extract_B7_features(b_df)
        features.update(b7_feat)
        
        # B8: 주의 지속능력 (탑승차량 찾기)
        b8_feat = extract_B8_features(b_df)
        features.update(b8_feat)
        
        # B9: 다중과제 (시각+청각)
        b9_feat = extract_B9_features(b_df)
        features.update(b9_feat)
        
        # B10: 다중과제 (시각+청각)
        b10_feat = extract_B10_features(b_df)
        features.update(b10_feat)
    
    return features

print("✅ extract_all_features_for_test_id 함수 정의 완료!")


✅ extract_all_features_for_test_id 함수 정의 완료!


In [17]:
def create_cognitive_profiles(feature_df: pd.DataFrame) -> pd.DataFrame:
    """
    인지 프로파일 그룹화
    
    Args:
        feature_df: 피처 DataFrame
    
    Returns:
        인지 프로파일이 추가된 DataFrame
    """
    df = feature_df.copy()
    
    # 지각운동요인 (A1~A5)
    perceptual_motor_cols = [c for c in df.columns if any(x in c for x in ['A1_', 'A2_', 'A3_', 'A4_', 'A5_'])]
    if len(perceptual_motor_cols) > 0:
        df['cognitive_perceptual_motor_mean'] = df[perceptual_motor_cols].mean(axis=1)
        df['cognitive_perceptual_motor_std'] = df[perceptual_motor_cols].std(axis=1)
    
    # 지적운동요인 (A6, A7)
    intellectual_motor_cols = [c for c in df.columns if any(x in c for x in ['A6_', 'A7_'])]
    if len(intellectual_motor_cols) > 0:
        df['cognitive_intellectual_motor_mean'] = df[intellectual_motor_cols].mean(axis=1)
        df['cognitive_intellectual_motor_std'] = df[intellectual_motor_cols].std(axis=1)
    
    # 정서/행동/스트레스 (A8, A9)
    emotional_cols = [c for c in df.columns if any(x in c for x in ['A8_', 'A9_'])]
    if len(emotional_cols) > 0:
        df['cognitive_emotional_mean'] = df[emotional_cols].mean(axis=1)
        df['cognitive_emotional_std'] = df[emotional_cols].std(axis=1)
    
    # B 검사 통합
    b_test_cols = [c for c in df.columns if c.startswith('B')]
    if len(b_test_cols) > 0:
        df['cognitive_b_test_mean'] = df[b_test_cols].mean(axis=1)
        df['cognitive_b_test_std'] = df[b_test_cols].std(axis=1)
    
    return df

print("✅ create_cognitive_profiles 함수 정의 완료!")


✅ create_cognitive_profiles 함수 정의 완료!


In [ ]:
def _process_single_test_id(args):
    """단일 Test_id 처리 함수 (병렬 처리용) - 메모리 최적화 버전"""
    # 미리 슬라이싱된 작은 DataFrame만 받음 (전체 DataFrame이 아닌 조각만)
    test_id, primary_key, label, test_type, df_A, df_B = args
    
    # 피처 추출
    features = extract_all_features_for_test_id(test_id, df_A, df_B)
    
    # 메타 정보 추가
    features['PrimaryKey'] = primary_key
    features['Label'] = label
    features['Test'] = test_type
    
    return features

def create_driver_level_features(train_meta: pd.DataFrame, train_A: pd.DataFrame, 
                                  train_B: pd.DataFrame, sample_size: int = None, 
                                  n_jobs: int = -1) -> pd.DataFrame:
    """
    Driver-level 피처 생성 (PrimaryKey 기준) - 병렬 처리 최적화 버전
    
    Args:
        train_meta: train.csv (Test_id, Test, Label 포함) - PrimaryKey는 이미 merge되어 있어야 함
        train_A: A 검사 데이터
        train_B: B 검사 데이터
        sample_size: 테스트용 샘플 크기 (None이면 전체)
        n_jobs: 병렬 처리 작업 수 (-1이면 모든 CPU 코어 사용)
    
    Returns:
        Driver-level 피처 DataFrame (각 행이 하나의 Test_id)
    """
    print("=" * 80)
    print("Driver-level 피처 생성 시작 (병렬 처리 최적화 버전)")
    print("=" * 80)
    
    # CPU 코어 수 확인
    if n_jobs == -1:
        n_jobs = multiprocessing.cpu_count()
    print(f"사용 CPU 코어 수: {n_jobs}개")
    
    # PrimaryKey가 없으면 추가
    if 'PrimaryKey' not in train_meta.columns:
        print("PrimaryKey 정보를 A/B 데이터에서 가져오는 중...")
        a_pk = train_A[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
        b_pk = train_B[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
        pk_df = pd.concat([a_pk, b_pk]).drop_duplicates(subset=['Test_id'])
        train_meta = train_meta.merge(pk_df, on='Test_id', how='left')
        print(f"  PrimaryKey 추가 완료: {train_meta['PrimaryKey'].notna().sum()}개 레코드")
    
    # PrimaryKey가 없는 레코드 제외
    train_meta = train_meta[train_meta['PrimaryKey'].notna()].copy()
    
    # 샘플링 (테스트용)
    if sample_size is not None:
        test_ids = train_meta['Test_id'].unique()[:sample_size]
        train_meta = train_meta[train_meta['Test_id'].isin(test_ids)]
        print(f"테스트용 샘플 크기: {len(test_ids)}개")
    
    total = len(train_meta)
    print(f"총 처리할 레코드 수: {total:,}개")
    
    # 최적화: 정렬 + 인덱스 맵 생성 (groupby보다 훨씬 빠름)
    print("데이터 인덱싱 최적화 중...")
    import time
    start_idx = time.time()
    
    # 1. Test_id로 정렬 (한 번만)
    print("  Step 1: 데이터 정렬 중...")
    train_A_sorted = train_A.sort_values('Test_id').reset_index(drop=True)
    train_B_sorted = train_B.sort_values('Test_id').reset_index(drop=True)
    print(f"    A 검사 정렬 완료: {len(train_A_sorted):,}개 행")
    print(f"    B 검사 정렬 완료: {len(train_B_sorted):,}개 행")
    
    # 2. Test_id별 시작/끝 인덱스 맵 생성 (numpy 최적화 버전)
    print("  Step 2: 인덱스 맵 생성 중...")
    train_A_idx_map = {}  # {test_id: (start_idx, end_idx)}
    train_B_idx_map = {}
    
    # A 검사 인덱스 맵 (numpy로 빠르게)
    if len(train_A_sorted) > 0:
        test_ids_A = train_A_sorted['Test_id'].values
        # 변화 지점 찾기 (diff 사용)
        diff_A = np.concatenate(([True], test_ids_A[1:] != test_ids_A[:-1]))
        change_indices_A = np.where(diff_A)[0]
        
        # 각 그룹의 시작/끝 인덱스
        print(f"    A 검사: {len(change_indices_A):,}개 고유 Test_id 발견")
        for i in range(len(change_indices_A)):
            start_idx = change_indices_A[i]
            end_idx = change_indices_A[i + 1] if i + 1 < len(change_indices_A) else len(test_ids_A)
            test_id = test_ids_A[start_idx]
            train_A_idx_map[test_id] = (int(start_idx), int(end_idx))
            
            # 진행 상황 출력 (10% 단위)
            if (i + 1) % max(1, len(change_indices_A) // 10) == 0:
                print(f"      A 검사 인덱스 맵: {i+1:,}/{len(change_indices_A):,} 처리됨")
    
    # B 검사 인덱스 맵 (numpy로 빠르게)
    if len(train_B_sorted) > 0:
        test_ids_B = train_B_sorted['Test_id'].values
        # 변화 지점 찾기
        diff_B = np.concatenate(([True], test_ids_B[1:] != test_ids_B[:-1]))
        change_indices_B = np.where(diff_B)[0]
        
        # 각 그룹의 시작/끝 인덱스
        print(f"    B 검사: {len(change_indices_B):,}개 고유 Test_id 발견")
        for i in range(len(change_indices_B)):
            start_idx = change_indices_B[i]
            end_idx = change_indices_B[i + 1] if i + 1 < len(change_indices_B) else len(test_ids_B)
            test_id = test_ids_B[start_idx]
            train_B_idx_map[test_id] = (int(start_idx), int(end_idx))
            
            # 진행 상황 출력 (10% 단위)
            if (i + 1) % max(1, len(change_indices_B) // 10) == 0:
                print(f"      B 검사 인덱스 맵: {i+1:,}/{len(change_indices_B):,} 처리됨")
    
    elapsed_idx = time.time() - start_idx
    print(f"  인덱싱 완료: {elapsed_idx:.1f}초 ({elapsed_idx/60:.1f}분)")
    print(f"  A 검사 인덱스 맵: {len(train_A_idx_map):,}개 Test_id")
    print(f"  B 검사 인덱스 맵: {len(train_B_idx_map):,}개 Test_id")
    
    # 3. 병렬 처리용 인자 준비 (메모리 최적화: 미리 슬라이싱)
    print("병렬 처리 준비 중... (데이터 미리 슬라이싱)")
    args_list = []
    
    # numpy 배열로 변환 (더 빠름)
    meta_values = train_meta[['Test_id', 'PrimaryKey', 'Label', 'Test']].values
    
    prep_start = time.time()
    for i, (test_id, primary_key, label, test_type) in enumerate(meta_values):
        # 진행 상황 출력 (5% 단위)
        if (i + 1) % max(1, total // 20) == 0:
            elapsed_prep = time.time() - prep_start
            progress = 100 * (i + 1) / total
            speed = (i + 1) / elapsed_prep if elapsed_prep > 0 else 0
            print(f"  진행: {i+1:,}/{total:,} ({progress:.1f}%) | 속도: {speed:.0f}개/초")
        
        # 💥 핵심 수정: 미리 슬라이싱하여 작은 조각만 args_list에 추가
        # 이렇게 하면 전체 DataFrame이 아닌 작은 조각만 메모리에 저장됨
        if test_id in train_A_idx_map:
            start_idx, end_idx = train_A_idx_map[test_id]
            # .copy()는 메모리 공유 문제를 피하기 위해 필요
            df_A = train_A_sorted.iloc[start_idx:end_idx].copy()
        else:
            df_A = pd.DataFrame()
        
        if test_id in train_B_idx_map:
            start_idx, end_idx = train_B_idx_map[test_id]
            df_B = train_B_sorted.iloc[start_idx:end_idx].copy()
        else:
            df_B = pd.DataFrame()
        
        # 작은 조각만 args_list에 추가 (메모리 효율적)
        args_list.append((
            test_id,
            primary_key,
            label,
            test_type,
            df_A,  # 전체 train_A_sorted 대신 작은 df_A 조각
            df_B   # 전체 train_B_sorted 대신 작은 df_B 조각
        ))
    
    elapsed_prep_total = time.time() - prep_start
    print(f"  준비 완료: {elapsed_prep_total:.1f}초")
    print(f"  메모리 최적화: 전체 DataFrame 대신 작은 조각만 저장됨")
    
    # 병렬 처리 실행
    print(f"병렬 처리 시작 (배치 크기: {max(1, total // (n_jobs * 10)):,}개)...")
    import time
    start_time = time.time()
    
    # joblib의 backend='threading' 대신 'loky' 사용 (더 안정적)
    # 배치 단위로 진행 상황 출력
    batch_size = max(1000, total // 100)  # 1% 단위로 출력
    results = []
    
    # 작은 배치로 나누어 처리 (메모리 효율성)
    for i in range(0, len(args_list), batch_size):
        batch = args_list[i:i+batch_size]
        batch_results = Parallel(n_jobs=n_jobs, verbose=0, backend='loky')(
            delayed(_process_single_test_id)(args) for args in batch
        )
        results.extend(batch_results)
        
        elapsed = time.time() - start_time
        processed = min(i + batch_size, total)
        progress = 100 * processed / total
        speed = processed / elapsed if elapsed > 0 else 0
        eta = (total - processed) / speed if speed > 0 else 0
        
        print(f"진행 중: {processed:,}/{total:,} ({progress:.1f}%) | "
              f"속도: {speed:.0f}개/초 | 경과: {elapsed:.1f}초 | 예상 남은 시간: {eta:.1f}초")
    
    elapsed_total = time.time() - start_time
    print(f"\n병렬 처리 완료! 총 소요 시간: {elapsed_total:.1f}초 ({elapsed_total/60:.1f}분)")
    print(f"평균 처리 속도: {total/elapsed_total:.0f}개/초")
    
    # DataFrame 변환
    print("DataFrame 변환 중...")
    feature_df = pd.DataFrame(results)
    
    print(f"\n피처 생성 완료: {feature_df.shape}")
    print(f"총 피처 수: {len(feature_df.columns) - 4}개 (메타 정보 제외)")
    
    return feature_df

print("✅ create_driver_level_features 함수 정의 완료!")


✅ create_driver_level_features 함수 정의 완료!


In [19]:
# 데이터 로드
DATA_DIR = "./data"

train_meta = pd.read_csv(f"{DATA_DIR}/train.csv")
train_A = pd.read_csv(f"{DATA_DIR}/train/A.csv")
train_B = pd.read_csv(f"{DATA_DIR}/train/B.csv")

print("=" * 80)
print("데이터 로딩 완료")
print("=" * 80)
print(f"train.csv: {train_meta.shape}")
print(f"train/A.csv: {train_A.shape}")
print(f"train/B.csv: {train_B.shape}")
print()

# 기본 정보 확인
print("[train.csv 컬럼]")
print(train_meta.columns.tolist())
print()

print("[train.csv 처음 5행]")
display(train_meta.head())

print("\n[train.csv 기본 통계]")
print(train_meta.describe())


데이터 로딩 완료
train.csv: (944767, 3)
train/A.csv: (647241, 37)
train/B.csv: (297526, 31)

[train.csv 컬럼]
['Test_id', 'Test', 'Label']

[train.csv 처음 5행]


,Test_id,Test,Label
0,0xE3EDFEA7DB8FF2606A19628967674BA957FB4BD58549...,A,0
1,0xDA572847455702C04D71C54677413CB8E31944B99289...,A,0
2,0xD5BB9FA4D3BC42EE494BD670F004564CB04A0DF8F819...,B,0
3,0x59D17D1C537B5FDE6622A9CB0A4192529CB4BD8D5422...,A,0
4,0x23005DA8BB4C84E1363A44A4248987798F3EAD4C58D3...,A,0



[train.csv 기본 통계]
               Label
count  944767.000000
mean        0.028877
std         0.167461
min         0.000000
25%         0.000000
50%         0.000000
75%         0.000000
max         1.000000


In [20]:
# PrimaryKey 기준으로 데이터 정리
print("=" * 80)
print("PrimaryKey 기준 데이터 정리")
print("=" * 80)

# train.csv에는 PrimaryKey가 없으므로 A.csv/B.csv에서 가져와야 함
# A.csv와 B.csv를 merge하여 PrimaryKey 정보 추출
print("[데이터 구조 확인]")
print(f"  - train.csv 컬럼: {train_meta.columns.tolist()}")
print(f"  - A.csv 컬럼 (처음 5개): {train_A.columns.tolist()[:5]}")
print(f"  - B.csv 컬럼 (처음 5개): {train_B.columns.tolist()[:5]}")
print()

# A.csv와 B.csv에서 PrimaryKey 추출 (Test_id별로 하나씩만)
a_pk = train_A[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
b_pk = train_B[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])

# train_meta에 PrimaryKey 추가 (A와 B 모두 merge)
train_meta = train_meta.merge(
    pd.concat([a_pk, b_pk]).drop_duplicates(subset=['Test_id']),
    on='Test_id',
    how='left'
)

print(f"[train.csv에 PrimaryKey 추가 후]")
print(f"  - train.csv 컬럼: {train_meta.columns.tolist()}")
print(f"  - PrimaryKey가 있는 레코드: {train_meta['PrimaryKey'].notna().sum()}개")
print(f"  - PrimaryKey가 없는 레코드: {train_meta['PrimaryKey'].isna().sum()}개")
print()

# PrimaryKey 기준 통계
print("[PrimaryKey 기준 통계]")
print(f"  - train.csv의 PrimaryKey 고유값: {train_meta['PrimaryKey'].nunique()}개")
print(f"  - train.csv의 총 레코드 수: {len(train_meta)}개")
print(f"  - A.csv의 PrimaryKey 고유값: {train_A['PrimaryKey'].nunique()}개")
print(f"  - A.csv의 총 레코드 수: {len(train_A)}개")
print(f"  - B.csv의 PrimaryKey 고유값: {train_B['PrimaryKey'].nunique()}개")
print(f"  - B.csv의 총 레코드 수: {len(train_B)}개")
print()

# Test_id와 PrimaryKey 관계 확인
print("[Test_id와 PrimaryKey 관계]")
print(f"  - train.csv의 Test_id 고유값: {train_meta['Test_id'].nunique()}개")
print(f"  - A.csv의 Test_id 고유값: {train_A['Test_id'].nunique()}개")
print(f"  - B.csv의 Test_id 고유값: {train_B['Test_id'].nunique()}개")
print()

# PrimaryKey별 검사 횟수 확인
pk_test_count = train_meta.groupby('PrimaryKey').size()
print(f"[PrimaryKey별 검사 횟수]")
print(f"  - 최소: {pk_test_count.min()}회")
print(f"  - 최대: {pk_test_count.max()}회")
print(f"  - 평균: {pk_test_count.mean():.2f}회")
print(f"  - 중앙값: {pk_test_count.median():.2f}회")
print()

# PrimaryKey별 검사 횟수 분포
print("[PrimaryKey별 검사 횟수 분포 (Top 20)]")
print(pk_test_count.value_counts().head(20))


PrimaryKey 기준 데이터 정리
[데이터 구조 확인]
  - train.csv 컬럼: ['Test_id', 'Test', 'Label']
  - A.csv 컬럼 (처음 5개): ['Test_id', 'Test', 'PrimaryKey', 'Age', 'TestDate']
  - B.csv 컬럼 (처음 5개): ['Test_id', 'Test', 'PrimaryKey', 'Age', 'TestDate']

[train.csv에 PrimaryKey 추가 후]
  - train.csv 컬럼: ['Test_id', 'Test', 'Label', 'PrimaryKey']
  - PrimaryKey가 있는 레코드: 944767개
  - PrimaryKey가 없는 레코드: 0개

[PrimaryKey 기준 통계]
  - train.csv의 PrimaryKey 고유값: 774242개
  - train.csv의 총 레코드 수: 944767개
  - A.csv의 PrimaryKey 고유값: 611862개
  - A.csv의 총 레코드 수: 647241개
  - B.csv의 PrimaryKey 고유값: 187592개
  - B.csv의 총 레코드 수: 297526개

[Test_id와 PrimaryKey 관계]
  - train.csv의 Test_id 고유값: 944767개
  - A.csv의 Test_id 고유값: 647241개
  - B.csv의 Test_id 고유값: 297526개

[PrimaryKey별 검사 횟수]
  - 최소: 1회
  - 최대: 10회
  - 평균: 1.22회
  - 중앙값: 1.00회

[PrimaryKey별 검사 횟수 분포 (Top 20)]
1     660596
2      72268
3      31068
4       6866
5       2105
6       1006
7        267
8         58
9          7
10         1
Name: count, dtype: int64


In [21]:
# 모델링: 불균형 처리 및 교차검증
import os

print("=" * 80)
print("모델링: 불균형 처리 및 교차검증")
print("=" * 80)

# all_features가 없으면 저장된 파일에서 로드
if 'all_features' not in globals() or all_features is None:
    output_path = "./driver_level_features.parquet"
    if os.path.exists(output_path):
        print(f"📂 저장된 피처 파일 로드: {output_path}")
        all_features = pd.read_parquet(output_path)
        print(f"✅ 로드 완료: {all_features.shape}")
    else:
        raise FileNotFoundError(f"피처 파일이 없습니다: {output_path}\nCell 48을 먼저 실행하세요.")

# 숫자형 피처만 선택 (이전 셀에서 정의되지 않았을 경우)
if 'numeric_cols' not in globals():
    numeric_cols = all_features.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ['Label', 'Test_id']]

# 데이터 준비
X = all_features[numeric_cols].fillna(all_features[numeric_cols].median())
y = all_features['Label'].values
primary_keys = all_features['PrimaryKey'].values if 'PrimaryKey' in all_features.columns else None

print(f"피처 수: {len(numeric_cols)}개")
print(f"샘플 수: {len(X)}개")
print(f"Label 분포: {np.bincount(y)}")
print(f"불균형 비율: {(y == 0).sum() / (y == 1).sum():.2f}:1")

# ============================================================
# 1. StratifiedKFold (라벨 분포 유지)
# ============================================================
print("\n[1] StratifiedKFold 교차검증 (라벨 분포 유지)")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
skf_aucs = []
skf_oof_preds = np.zeros(len(y))

scale_pos_weight = (y == 0).sum() / (y == 1).sum()

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y[train_idx], y[valid_idx]
    
    # LightGBM 학습
    train_data = lgb.Dataset(X_train, label=y_train)
    valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
    
    model = lgb.train(
        lgb_params,
        train_data,
        num_boost_round=1000,
        valid_sets=[valid_data],
        callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
    )
    
    # 예측
    y_pred = model.predict(X_valid)
    skf_oof_preds[valid_idx] = y_pred
    
    # 평가
    auc = roc_auc_score(y_valid, y_pred)
    skf_aucs.append(auc)
    print(f"  Fold {fold}: AUC = {auc:.4f}")

print(f"\n  평균 AUC: {np.mean(skf_aucs):.4f} ± {np.std(skf_aucs):.4f}")
print(f"  최고 AUC: {np.max(skf_aucs):.4f}")
print(f"  최저 AUC: {np.min(skf_aucs):.4f}")

# ============================================================
# 2. GroupKFold (PrimaryKey 기준)
# ============================================================
if primary_keys is not None:
    print("\n[2] GroupKFold 교차검증 (PrimaryKey 기준)")
    
    gkf = GroupKFold(n_splits=5)
    gkf_aucs = []
    gkf_oof_preds = np.zeros(len(y))
    
    for fold, (train_idx, valid_idx) in enumerate(gkf.split(X, y, groups=primary_keys), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]
        
        # 중복 PrimaryKey 확인
        train_pks = set(primary_keys[train_idx])
        valid_pks = set(primary_keys[valid_idx])
        overlap = train_pks & valid_pks
        print(f"  Fold {fold}: Train PKs={len(train_pks)}, Valid PKs={len(valid_pks)}, Overlap={len(overlap)}")
        
        # LightGBM 학습
        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
        
        model = lgb.train(
            lgb_params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
        )
        
        # 예측
        y_pred = model.predict(X_valid)
        gkf_oof_preds[valid_idx] = y_pred
        
        # 평가
        auc = roc_auc_score(y_valid, y_pred)
        gkf_aucs.append(auc)
        print(f"    AUC = {auc:.4f}")
    
    print(f"\n  평균 AUC: {np.mean(gkf_aucs):.4f} ± {np.std(gkf_aucs):.4f}")
    print(f"  최고 AUC: {np.max(gkf_aucs):.4f}")
    print(f"  최저 AUC: {np.min(gkf_aucs):.4f}")
else:
    print("\n[2] GroupKFold 스킵 (PrimaryKey 정보 없음)")
    gkf_aucs = []

# ============================================================
# 3. ROC 곡선 비교
# ============================================================
print("\n[3] ROC 곡선 비교")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# StratifiedKFold ROC
fpr_skf, tpr_skf, _ = roc_curve(y, skf_oof_preds)
auc_skf = roc_auc_score(y, skf_oof_preds)

axes[0].plot(fpr_skf, tpr_skf, label=f'StratifiedKFold (AUC = {auc_skf:.4f})', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', label='Random')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curve - StratifiedKFold', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# GroupKFold ROC
if len(gkf_aucs) > 0:
    fpr_gkf, tpr_gkf, _ = roc_curve(y, gkf_oof_preds)
    auc_gkf = roc_auc_score(y, gkf_oof_preds)
    
    axes[1].plot(fpr_gkf, tpr_gkf, label=f'GroupKFold (AUC = {auc_gkf:.4f})', linewidth=2, color='orange')
    axes[1].plot([0, 1], [0, 1], 'k--', label='Random')
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title('ROC Curve - GroupKFold', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'GroupKFold 결과 없음', ha='center', va='center')
    axes[1].set_title('ROC Curve - GroupKFold', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

# ============================================================
# 4. Calibration Curve
# ============================================================
print("\n[4] Calibration Curve")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# StratifiedKFold
fraction_of_positives_skf, mean_predicted_value_skf = calibration_curve(
    y, skf_oof_preds, n_bins=10
)

axes[0].plot(mean_predicted_value_skf, fraction_of_positives_skf, 's-', label='StratifiedKFold')
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
axes[0].set_xlabel('Mean Predicted Probability')
axes[0].set_ylabel('Fraction of Positives')
axes[0].set_title('Calibration Curve - StratifiedKFold', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# GroupKFold
if len(gkf_aucs) > 0:
    fraction_of_positives_gkf, mean_predicted_value_gkf = calibration_curve(
        y, gkf_oof_preds, n_bins=10
    )
    
    axes[1].plot(mean_predicted_value_gkf, fraction_of_positives_gkf, 's-', 
                label='GroupKFold', color='orange')
    axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
    axes[1].set_xlabel('Mean Predicted Probability')
    axes[1].set_ylabel('Fraction of Positives')
    axes[1].set_title('Calibration Curve - GroupKFold', fontsize=12, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'GroupKFold 결과 없음', ha='center', va='center')
    axes[1].set_title('Calibration Curve - GroupKFold', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✅ 모델링 완료!")


모델링: 불균형 처리 및 교차검증


FileNotFoundError: 피처 파일이 없습니다: ./driver_level_features.parquet
Cell 48을 먼저 실행하세요.

In [ ]:
# 라벨별 피처 분포 비교 (인지 프로파일 중심)
print("=" * 80)
print("라벨별 인지 프로파일 비교")
print("=" * 80)

if 'Label' in sample_features.columns:
    cognitive_cols = [c for c in sample_features.columns if c.startswith('cognitive_')]
    
    if len(cognitive_cols) > 0:
        # 라벨별 평균 비교
        label_stats = sample_features.groupby('Label')[cognitive_cols].mean()
        print("\n[라벨별 인지 프로파일 평균]")
        display(label_stats)
        
        # 시각화
        n_cols = len(cognitive_cols)
        n_rows = (n_cols + 2) // 3
        
        if n_cols > 0:
            fig, axes = plt.subplots(n_rows, 3, figsize=(18, 6*n_rows))
            axes = axes.flatten() if n_cols > 1 else [axes]
            
            for idx, col in enumerate(cognitive_cols[:len(axes)]):
                ax = axes[idx]
                
                # 라벨별 분포
                label_0 = sample_features[sample_features['Label'] == 0][col].dropna()
                label_1 = sample_features[sample_features['Label'] == 1][col].dropna()
                
                if len(label_0) > 0 and len(label_1) > 0:
                    ax.hist(label_0, alpha=0.5, label='Label 0', bins=20, color='skyblue')
                    ax.hist(label_1, alpha=0.5, label='Label 1', bins=20, color='salmon')
                    ax.set_title(f'{col}', fontsize=10, fontweight='bold')
                    ax.set_xlabel('Value')
                    ax.set_ylabel('Frequency')
                    ax.legend()
                    ax.grid(alpha=0.3)
            
            # 빈 subplot 제거
            for idx in range(len(cognitive_cols), len(axes)):
                fig.delaxes(axes[idx])
            
            plt.tight_layout()
            plt.show()
    
    # 주요 검사별 지표 비교
    print("\n[주요 검사별 지표 - 라벨별 평균]")
    key_features = [
        'A1_acc_overall', 'A1_acc_slow_fast_diff',
        'A3_switch_cost_rt', 'A3_peripheral_deficit',
        'A4_stroop_rt_effect', 'A4_stroop_acc_effect',
        'A5_hit_rate', 'A5_false_alarm_rate',
        'B1_hit_rate', 'B3_acc', 'B4_flanker_rt_effect',
        'B8_sustained_attention_deficit',
        'B9_aud_hit_rate', 'B10_aud_hit_rate'
    ]
    
    available_key_features = [f for f in key_features if f in sample_features.columns]
    if len(available_key_features) > 0:
        key_stats = sample_features.groupby('Label')[available_key_features].mean()
        display(key_stats)
        
        # 박스플롯
        n_key = len(available_key_features)
        n_rows_key = (n_key + 2) // 3
        
        if n_key > 0:
            fig, axes = plt.subplots(n_rows_key, 3, figsize=(18, 6*n_rows_key))
            axes = axes.flatten() if n_key > 1 else [axes]
            
            for idx, col in enumerate(available_key_features[:len(axes)]):
                ax = axes[idx]
                
                data_to_plot = [
                    sample_features[sample_features['Label'] == 0][col].dropna().values,
                    sample_features[sample_features['Label'] == 1][col].dropna().values
                ]
                
                bp = ax.boxplot(data_to_plot, labels=['Label 0', 'Label 1'], patch_artist=True)
                bp['boxes'][0].set_facecolor('skyblue')
                bp['boxes'][1].set_facecolor('salmon')
                ax.set_title(f'{col}', fontsize=10, fontweight='bold')
                ax.set_ylabel('Value')
                ax.grid(alpha=0.3, axis='y')
            
            for idx in range(len(available_key_features), len(axes)):
                fig.delaxes(axes[idx])
            
            plt.tight_layout()
            plt.show()
else:
    print("Label 컬럼이 없습니다. 전체 데이터로 피처 생성 후 다시 시도하세요.")


라벨별 인지 프로파일 비교


NameError: name 'sample_features' is not defined

In [ ]:
# 전체 데이터로 피처 생성 (옵션 - 시간이 오래 걸릴 수 있음)
# 주의: 전체 데이터는 약 94만 개이므로 시간이 매우 오래 걸릴 수 있습니다.
# 필요시에만 실행하세요.

# 전체 피처 생성
# print("=" * 80)
# print("전체 데이터로 피처 생성 시작")
# print("=" * 80)
# 
# # PrimaryKey 추가 확인
# if 'PrimaryKey' not in train_meta.columns:
#     a_pk = train_A[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
#     b_pk = train_B[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
#     pk_df = pd.concat([a_pk, b_pk]).drop_duplicates(subset=['Test_id'])
#     train_meta = train_meta.merge(pk_df, on='Test_id', how='left')
#     train_meta = train_meta[train_meta['PrimaryKey'].notna()].copy()
# 
# # 전체 피처 생성
# all_features = create_driver_level_features(train_meta, train_A, train_B, sample_size=None)
# 
# # 인지 프로파일 적용
# all_features = create_cognitive_profiles(all_features)
# 
# # 저장
# output_path = "./driver_level_features.parquet"
# all_features.to_parquet(output_path, index=False)
# print(f"\n피처 저장 완료: {output_path}")
# print(f"Shape: {all_features.shape}")

print("전체 데이터 피처 생성은 주석 처리되어 있습니다.")
print("필요시 위 주석을 해제하고 실행하세요.")


전체 데이터 피처 생성은 주석 처리되어 있습니다.
필요시 위 주석을 해제하고 실행하세요.


In [1]:
# 전체 데이터로 피처 생성 (저장된 파일이 있으면 로드, 없으면 생성)
import os

print("=" * 80)
print("전체 데이터 피처 생성/로드")
print("=" * 80)

output_path = "./driver_level_features.parquet"

# 저장된 파일이 있으면 로드
if os.path.exists(output_path):
    print(f"\n✅ 저장된 피처 파일 발견: {output_path}")
    print("   파일을 로드합니다...")
    all_features = pd.read_parquet(output_path)
    print(f"\n✅ 피처 로드 완료!")
    print(f"  Shape: {all_features.shape}")
    print(f"  컬럼 수: {len(all_features.columns)}개")
    print(f"\n  Label 분포:")
    print(all_features['Label'].value_counts())
    print(f"\n  Label 비율:")
    print(all_features['Label'].value_counts(normalize=True))
    print(f"\n  PrimaryKey 수: {all_features['PrimaryKey'].nunique()}개")
    print(f"  Test A: {(all_features['Test'] == 'A').sum()}개")
    print(f"  Test B: {(all_features['Test'] == 'B').sum()}개")
    print("\n💡 새로 생성하려면 파일을 삭제하고 다시 실행하세요.")
else:
    print("\n📝 저장된 파일이 없습니다. 피처를 생성합니다...")
    
    # 함수 정의 확인
    required_functions = ['create_driver_level_features', 'create_cognitive_profiles', 'extract_all_features_for_test_id']
    missing_functions = [f for f in required_functions if f not in globals()]
    
    if missing_functions:
        print("⚠️  필요한 함수가 정의되지 않았습니다!")
        print(f"   누락된 함수: {missing_functions}")
        print("\n다음 셀들을 먼저 실행하세요:")
        print("  - Cell 38: create_cognitive_profiles")
        print("  - Cell 66: extract_all_features_for_test_id")
        print("  - Cell 68: create_driver_level_features")
        raise NameError(f"함수가 정의되지 않았습니다: {missing_functions}")
    
    # PrimaryKey 추가 확인
    if 'PrimaryKey' not in train_meta.columns:
        print("PrimaryKey 추가 중...")
        a_pk = train_A[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
        b_pk = train_B[['Test_id', 'PrimaryKey']].drop_duplicates(subset=['Test_id'])
        pk_df = pd.concat([a_pk, b_pk]).drop_duplicates(subset=['Test_id'])
        train_meta = train_meta.merge(pk_df, on='Test_id', how='left')
        train_meta = train_meta[train_meta['PrimaryKey'].notna()].copy()
    
    print(f"\n전체 데이터 크기: {len(train_meta)}개 레코드")
    
    # CPU 코어 수 확인
    n_cores = multiprocessing.cpu_count()
    print(f"사용 가능한 CPU 코어: {n_cores}개")
    print(f"병렬 처리로 예상 속도 향상: 약 {n_cores}배")
    
    # 예상 소요 시간 계산 (병렬 처리 기준)
    # 기존: 2분에 2000개 = 0.6개/초
    # 병렬 처리 후: 0.6 * n_cores = 약 5-10개/초
    estimated_speed = 0.6 * n_cores  # 개/초
    estimated_time_min = len(train_meta) / estimated_speed / 60
    print(f"예상 소요 시간: 약 {estimated_time_min:.1f}분 ({estimated_time_min/60:.1f}시간) - 병렬 처리 기준")
    print("\n피처 생성 시작... (진행 상황이 출력됩니다)")
    print("💡 n_jobs=-1 (기본값): 모든 CPU 코어 사용")
    print("💡 n_jobs=4: 4개 코어만 사용 (메모리 부족 시)")
    
    # 전체 피처 생성 (병렬 처리 최적화 버전)
    # n_jobs=-1: 모든 CPU 코어 사용 (기본값)
    # n_jobs=4: 4개 코어만 사용 (메모리 부족 시 조정)
    all_features = create_driver_level_features(
        train_meta, train_A, train_B, 
        sample_size=None,
        n_jobs=4  # 모든 CPU 코어 사용 (-1), 또는 특정 개수 지정 (예: 4, 8)
    )
    
    # 인지 프로파일 적용
    print("\n인지 프로파일 그룹화 적용 중...")
    all_features = create_cognitive_profiles(all_features)
    
    print(f"\n✅ 피처 생성 완료!")
    print(f"  Shape: {all_features.shape}")
    print(f"  컬럼 수: {len(all_features.columns)}개")
    
    # 저장
    all_features.to_parquet(output_path, index=False)
    print(f"\n💾 피처 저장 완료: {output_path}")
    
    # 기본 통계
    print("\n[기본 통계]")
    print(f"  Label 분포:")
    print(all_features['Label'].value_counts())
    print(f"\n  Label 비율:")
    print(all_features['Label'].value_counts(normalize=True))
    print(f"\n  PrimaryKey 수: {all_features['PrimaryKey'].nunique()}개")
    print(f"  Test A: {(all_features['Test'] == 'A').sum()}개")
    print(f"  Test B: {(all_features['Test'] == 'B').sum()}개")


전체 데이터 피처 생성/로드

📝 저장된 파일이 없습니다. 피처를 생성합니다...
⚠️  필요한 함수가 정의되지 않았습니다!
   누락된 함수: ['create_driver_level_features', 'create_cognitive_profiles', 'extract_all_features_for_test_id']

다음 셀들을 먼저 실행하세요:
  - Cell 38: create_cognitive_profiles
  - Cell 66: extract_all_features_for_test_id
  - Cell 68: create_driver_level_features


NameError: 함수가 정의되지 않았습니다: ['create_driver_level_features', 'create_cognitive_profiles', 'extract_all_features_for_test_id']

In [ ]:
# 요약 및 다음 단계
print("=" * 80)
print("분석 결과 요약")
print("=" * 80)

print("\n[완료된 작업]")
print("  ✅ 전체 데이터로 피처 생성")
print("  ✅ PCA/UMAP 시각화")
print("  ✅ 피처 중요도 분석 (LightGBM, SHAP)")
print("  ✅ 모델링 (StratifiedKFold, GroupKFold)")
print("  ✅ 인지 프로파일 기반 분석")

print("\n[주요 발견 사항]")

# 1. 피처 중요도 Top 10
if 'feature_importance_df' in locals():
    top_10_features = feature_importance_df.head(10).index.tolist()
    print(f"\n  1. 가장 중요한 피처 Top 10:")
    for idx, feat in enumerate(top_10_features, 1):
        print(f"     {idx:2d}. {feat}")

# 2. 모델 성능
if 'skf_aucs' in locals():
    print(f"\n  2. 모델 성능 (StratifiedKFold):")
    print(f"     평균 AUC: {np.mean(skf_aucs):.4f} ± {np.std(skf_aucs):.4f}")

if 'gkf_aucs' in locals() and len(gkf_aucs) > 0:
    print(f"\n  3. 모델 성능 (GroupKFold):")
    print(f"     평균 AUC: {np.mean(gkf_aucs):.4f} ± {np.std(gkf_aucs):.4f}")

# 3. 인지 프로파일
if 'coef_df' in locals():
    top_positive = coef_df[coef_df['coefficient'] > 0].head(3)
    top_negative = coef_df[coef_df['coefficient'] < 0].head(3)
    
    print(f"\n  4. 인지 프로파일 영향:")
    if len(top_positive) > 0:
        print(f"     위험도 증가 요인:")
        for _, row in top_positive.iterrows():
            print(f"       - {row['feature']}: {row['coefficient']:.4f}")
    if len(top_negative) > 0:
        print(f"     위험도 감소 요인:")
        for _, row in top_negative.iterrows():
            print(f"       - {row['feature']}: {row['coefficient']:.4f}")

print("\n[다음 단계 제안]")
print("  1. 하이퍼파라미터 튜닝")
print("     - LightGBM 파라미터 최적화")
print("     - Optuna 등으로 자동 튜닝")
print("")
print("  2. 피처 엔지니어링")
print("     - 중요도 높은 피처 기반 상호작용 피처 생성")
print("     - 인지 프로파일 조합 피처 생성")
print("")
print("  3. 앙상블")
print("     - 여러 모델 조합 (LightGBM, XGBoost, CatBoost)")
print("     - Stacking/Blending")
print("")
print("  4. 시퀀스 모델 활용")
print("     - GRU/LSTM으로 시간 이력 모델링")
print("     - Tabular + Sequence 하이브리드")
print("")
print("  5. 불균형 처리 개선")
print("     - SMOTE, ADASYN 등 오버샘플링")
print("     - Focal Loss 등 클래스 불균형 특화 손실 함수")

print("\n✅ 전체 분석 완료!")
print(f"💾 피처 파일: driver_level_features.parquet")
print(f"📊 분석 결과는 위 셀들의 출력을 참고하세요.")


분석 결과 요약

[완료된 작업]
  ✅ 전체 데이터로 피처 생성
  ✅ PCA/UMAP 시각화
  ✅ 피처 중요도 분석 (LightGBM, SHAP)
  ✅ 모델링 (StratifiedKFold, GroupKFold)
  ✅ 인지 프로파일 기반 분석

[주요 발견 사항]

[다음 단계 제안]
  1. 하이퍼파라미터 튜닝
     - LightGBM 파라미터 최적화
     - Optuna 등으로 자동 튜닝

  2. 피처 엔지니어링
     - 중요도 높은 피처 기반 상호작용 피처 생성
     - 인지 프로파일 조합 피처 생성

  3. 앙상블
     - 여러 모델 조합 (LightGBM, XGBoost, CatBoost)
     - Stacking/Blending

  4. 시퀀스 모델 활용
     - GRU/LSTM으로 시간 이력 모델링
     - Tabular + Sequence 하이브리드

  5. 불균형 처리 개선
     - SMOTE, ADASYN 등 오버샘플링
     - Focal Loss 등 클래스 불균형 특화 손실 함수

✅ 전체 분석 완료!
💾 피처 파일: driver_level_features.parquet
📊 분석 결과는 위 셀들의 출력을 참고하세요.


In [ ]:
# 상관관계 분석
print("=" * 80)
print("상관관계 분석")
print("=" * 80)

# 숫자형 컬럼만 선택
numeric_cols = sample_features.select_dtypes(include=[np.number]).columns.tolist()
# 메타 정보 제외
numeric_cols = [c for c in numeric_cols if c not in ['Label', 'Test_id']]

if len(numeric_cols) > 0:
    # Label과의 상관관계
    if 'Label' in sample_features.columns:
        correlations = sample_features[numeric_cols + ['Label']].corr()['Label'].sort_values(ascending=False)
        correlations = correlations[correlations.index != 'Label']
        
        print("\n[Label과의 상관계수 (절대값 기준 Top 20)]")
        top_corr = correlations.abs().sort_values(ascending=False).head(20)
        for idx, val in top_corr.items():
            orig_corr = correlations[idx]
            print(f"  {idx:40s}: {orig_corr:7.4f}")
        
        # 상관관계 시각화 (인지 프로파일 + 주요 지표)
        cognitive_and_key = [c for c in numeric_cols if c.startswith('cognitive_') or 
                            any(x in c for x in ['stroop', 'switch_cost', 'hit_rate', 'flanker'])]
        
        if len(cognitive_and_key) > 1:
            # 상관관계 행렬
            corr_matrix = sample_features[cognitive_and_key + ['Label']].corr()
            
            plt.figure(figsize=(14, 12))
            sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
            plt.title('인지 프로파일 및 주요 지표 상관관계 행렬', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
        
        # 인지 프로파일 간 상관관계
        cognitive_cols_only = [c for c in numeric_cols if c.startswith('cognitive_')]
        if len(cognitive_cols_only) > 1:
            print("\n[인지 프로파일 간 상관계수]")
            cognitive_corr = sample_features[cognitive_cols_only].corr()
            display(cognitive_corr)
            
            plt.figure(figsize=(10, 8))
            sns.heatmap(cognitive_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
                       square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
            plt.title('인지 프로파일 간 상관관계', fontsize=14, fontweight='bold')
            plt.tight_layout()
            plt.show()
else:
    print("분석할 숫자형 컬럼이 없습니다.")


상관관계 분석


NameError: name 'sample_features' is not defined

In [ ]:
# PCA/UMAP 시각화
import os

print("=" * 80)
print("PCA/UMAP 시각화")
print("=" * 80)

# all_features가 없으면 저장된 파일에서 로드
if 'all_features' not in globals() or all_features is None:
    output_path = "./driver_level_features.parquet"
    if os.path.exists(output_path):
        print(f"📂 저장된 피처 파일 로드: {output_path}")
        all_features = pd.read_parquet(output_path)
        print(f"✅ 로드 완료: {all_features.shape}")
    else:
        raise FileNotFoundError(f"피처 파일이 없습니다: {output_path}\nCell 48을 먼저 실행하세요.")

# 숫자형 피처만 선택
numeric_cols = all_features.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['Label', 'Test_id']]

# 결측치 처리 (평균으로 대체)
X_viz = all_features[numeric_cols].fillna(all_features[numeric_cols].mean())
y_viz = all_features['Label'].values

print(f"시각화용 피처 수: {len(numeric_cols)}개")
print(f"샘플 수: {len(X_viz)}개")

# 스케일링
scaler_viz = StandardScaler()
X_viz_scaled = scaler_viz.fit_transform(X_viz)

# ============================================================
# 1. PCA (2D)
# ============================================================
print("\n[1] PCA (2D) 적용 중...")
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_viz_scaled)

print(f"  설명된 분산 비율: {pca.explained_variance_ratio_.sum():.4f}")
print(f"  PC1: {pca.explained_variance_ratio_[0]:.4f}")
print(f"  PC2: {pca.explained_variance_ratio_[1]:.4f}")

# PCA 시각화
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 라벨별 색칠
for label, color, name in [(0, 'skyblue', 'Label 0 (Normal)'), (1, 'salmon', 'Label 1 (Risk)')]:
    mask = y_viz == label
    axes[0].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=color, alpha=0.5, s=10, label=name)

axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
axes[0].set_title('PCA 2D Visualization', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# 밀도 플롯
for label, color, name in [(0, 'skyblue', 'Label 0'), (1, 'salmon', 'Label 1')]:
    mask = y_viz == label
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], 
                   c=color, alpha=0.3, s=5, label=name)

axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)')
axes[1].set_title('PCA 2D - Density View', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# ============================================================
# 2. UMAP (2D) - 선택적
# ============================================================
if HAS_UMAP:
    print("\n[2] UMAP (2D) 적용 중... (시간이 다소 걸릴 수 있습니다)")
    
    # 샘플링 (UMAP는 시간이 오래 걸리므로 샘플링)
    n_samples_umap = min(10000, len(X_viz_scaled))
    if n_samples_umap < len(X_viz_scaled):
        print(f"  샘플링: {n_samples_umap}개 (전체 {len(X_viz_scaled)}개 중)")
        indices = np.random.choice(len(X_viz_scaled), n_samples_umap, replace=False)
        X_umap_input = X_viz_scaled[indices]
        y_umap_input = y_viz[indices]
    else:
        X_umap_input = X_viz_scaled
        y_umap_input = y_viz
    
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    X_umap = reducer.fit_transform(X_umap_input)
    
    # UMAP 시각화
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for label, color, name in [(0, 'skyblue', 'Label 0 (Normal)'), (1, 'salmon', 'Label 1 (Risk)')]:
        mask = y_umap_input == label
        axes[0].scatter(X_umap[mask, 0], X_umap[mask, 1], 
                       c=color, alpha=0.5, s=10, label=name)
    
    axes[0].set_xlabel('UMAP 1')
    axes[0].set_ylabel('UMAP 2')
    axes[0].set_title('UMAP 2D Visualization', fontsize=14, fontweight='bold')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # 밀도 플롯
    for label, color, name in [(0, 'skyblue', 'Label 0'), (1, 'salmon', 'Label 1')]:
        mask = y_umap_input == label
        axes[1].scatter(X_umap[mask, 0], X_umap[mask, 1], 
                       c=color, alpha=0.3, s=5, label=name)
    
    axes[1].set_xlabel('UMAP 1')
    axes[1].set_ylabel('UMAP 2')
    axes[1].set_title('UMAP 2D - Density View', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("\n[2] UMAP 스킵 (라이브러리 없음)")

# ============================================================
# 3. 인지 프로파일별 PCA
# ============================================================
print("\n[3] 인지 프로파일별 PCA 시각화...")
cognitive_cols = [c for c in all_features.columns if c.startswith('cognitive_')]

if len(cognitive_cols) > 0:
    X_cognitive = all_features[cognitive_cols].fillna(all_features[cognitive_cols].mean())
    X_cognitive_scaled = StandardScaler().fit_transform(X_cognitive)
    
    pca_cognitive = PCA(n_components=2, random_state=42)
    X_cognitive_pca = pca_cognitive.fit_transform(X_cognitive_scaled)
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for label, color, name in [(0, 'skyblue', 'Label 0 (Normal)'), (1, 'salmon', 'Label 1 (Risk)')]:
        mask = y_viz == label
        ax.scatter(X_cognitive_pca[mask, 0], X_cognitive_pca[mask, 1], 
                  c=color, alpha=0.5, s=10, label=name)
    
    ax.set_xlabel(f'PC1 ({pca_cognitive.explained_variance_ratio_[0]:.2%} variance)')
    ax.set_ylabel(f'PC2 ({pca_cognitive.explained_variance_ratio_[1]:.2%} variance)')
    ax.set_title('PCA 2D - Cognitive Profiles Only', fontsize=14, fontweight='bold')
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"  인지 프로파일 PCA 설명 분산: {pca_cognitive.explained_variance_ratio_.sum():.4f}")
else:
    print("  인지 프로파일 컬럼이 없습니다.")


PCA/UMAP 시각화


FileNotFoundError: 피처 파일이 없습니다: ./driver_level_features.parquet
Cell 48을 먼저 실행하세요.

In [ ]:
# 피처 중요도 분석
import os

print("=" * 80)
print("피처 중요도 분석")
print("=" * 80)

# all_features가 없으면 저장된 파일에서 로드
if 'all_features' not in globals() or all_features is None:
    output_path = "./driver_level_features.parquet"
    if os.path.exists(output_path):
        print(f"📂 저장된 피처 파일 로드: {output_path}")
        all_features = pd.read_parquet(output_path)
        print(f"✅ 로드 완료: {all_features.shape}")
    else:
        raise FileNotFoundError(f"피처 파일이 없습니다: {output_path}\nCell 48을 먼저 실행하세요.")

# 숫자형 피처만 선택 (이전 셀에서 정의되지 않았을 경우)
if 'numeric_cols' not in globals():
    numeric_cols = all_features.select_dtypes(include=[np.number]).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c not in ['Label', 'Test_id']]

# 데이터 준비
X = all_features[numeric_cols].fillna(all_features[numeric_cols].median())
y = all_features['Label'].values

print(f"피처 수: {len(numeric_cols)}개")
print(f"샘플 수: {len(X)}개")
print(f"Label 분포: {np.bincount(y)}")

# ============================================================
# 1. 모델 학습 및 피처 중요도 분석
# ============================================================
if HAS_LIGHTGBM:
    print("\n[1] LightGBM 모델 학습 및 피처 중요도 분석...")
    
    # 불균형 처리
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    print(f"  scale_pos_weight: {scale_pos_weight:.2f}")
    
    # LightGBM 파라미터
    lgb_params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'num_leaves': 31,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'scale_pos_weight': scale_pos_weight,
        'random_state': 42,
        'verbose': -1
    }
    
    # StratifiedKFold로 학습
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    feature_importance_df = pd.DataFrame(index=numeric_cols)
    fold_aucs = []
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]
        
        # LightGBM 데이터셋
        train_data = lgb.Dataset(X_train, label=y_train)
        valid_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)
        
        # 학습
        model = lgb.train(
            lgb_params,
            train_data,
            num_boost_round=1000,
            valid_sets=[valid_data],
            callbacks=[lgb.early_stopping(stopping_rounds=50), lgb.log_evaluation(0)]
        )
        
        # 예측 및 평가
        y_pred = model.predict(X_valid)
        auc = roc_auc_score(y_valid, y_pred)
        fold_aucs.append(auc)
        print(f"  Fold {fold}: AUC = {auc:.4f}")
        
        # 피처 중요도
        importance = model.feature_importance(importance_type='gain')
        feature_importance_df[f'fold_{fold}'] = importance
    
    # 평균 피처 중요도
    feature_importance_df['mean_importance'] = feature_importance_df[[f'fold_{fold}' for fold in range(1, 6)]].mean(axis=1)
    feature_importance_df = feature_importance_df.sort_values('mean_importance', ascending=False)
    
    print(f"\n  평균 AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
    
    model_name = "LightGBM"
    
else:
    print("\n[1] HistGradientBoostingClassifier 모델 학습 및 피처 중요도 분석...")
    print("  (LightGBM이 없어서 sklearn의 HistGradientBoostingClassifier 사용)")
    
    # 불균형 처리
    scale_pos_weight = (y == 0).sum() / (y == 1).sum()
    print(f"  scale_pos_weight: {scale_pos_weight:.2f}")
    
    # StratifiedKFold로 학습
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    feature_importance_df = pd.DataFrame(index=numeric_cols)
    fold_aucs = []
    
    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), 1):
        X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
        y_train, y_valid = y[train_idx], y[valid_idx]
        
        # HistGradientBoostingClassifier 학습
        model = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=200,
            max_leaf_nodes=31,
            random_state=42,
            class_weight='balanced'  # 불균형 처리
        )
        model.fit(X_train, y_train)
        
        # 예측 및 평가
        y_pred = model.predict_proba(X_valid)[:, 1]
        auc = roc_auc_score(y_valid, y_pred)
        fold_aucs.append(auc)
        print(f"  Fold {fold}: AUC = {auc:.4f}")
        
        # 피처 중요도
        importance = model.feature_importances_
        feature_importance_df[f'fold_{fold}'] = importance
    
    # 평균 피처 중요도
    feature_importance_df['mean_importance'] = feature_importance_df[[f'fold_{fold}' for fold in range(1, 6)]].mean(axis=1)
    feature_importance_df = feature_importance_df.sort_values('mean_importance', ascending=False)
    
    print(f"\n  평균 AUC: {np.mean(fold_aucs):.4f} ± {np.std(fold_aucs):.4f}")
    
    model_name = "HistGradientBoosting"

# Top 20 피처 중요도 시각화
top_n = 20
top_features = feature_importance_df.head(top_n)

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(range(len(top_features)), top_features['mean_importance'], color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features.index)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importance ({model_name})', fontsize=14, fontweight='bold')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

# Top 피처 목록
print(f"\n[Top {top_n} 피처]")
for idx, (feat, imp) in enumerate(top_features['mean_importance'].items(), 1):
    print(f"  {idx:2d}. {feat:40s}: {imp:10.2f}")

# ============================================================
# 2. SHAP 값 분석 (선택적)
# ============================================================
if HAS_SHAP and HAS_LIGHTGBM:
    print("\n[2] SHAP 값 분석 (샘플 데이터 사용)...")
    
    # 마지막 fold 모델 사용
    train_data_final = lgb.Dataset(X, label=y)
    model_final = lgb.train(
        lgb_params,
        train_data_final,
        num_boost_round=500,
        callbacks=[lgb.log_evaluation(0)]
    )
    
    # SHAP 값 계산 (샘플링하여 시간 단축)
    n_shap_samples = min(500, len(X))
    shap_indices = np.random.choice(len(X), n_shap_samples, replace=False)
    X_shap = X.iloc[shap_indices]
    
    print(f"  SHAP 계산 중... (샘플 {n_shap_samples}개)")
    explainer = shap.TreeExplainer(model_final)
    shap_values = explainer.shap_values(X_shap)
    
    # SHAP summary plot
    print("  SHAP summary plot 생성 중...")
    
    # Summary plot (bar)
    plt.figure(figsize=(10, 8))
    shap.summary_plot(shap_values[1], X_shap, plot_type="bar", show=False, max_display=20)
    plt.title('SHAP Feature Importance (Top 20)', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Summary plot (dot)
    plt.figure(figsize=(12, 8))
    shap.summary_plot(shap_values[1], X_shap, plot_type="dot", show=False, max_display=15)
    plt.title('SHAP Summary Plot', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # 인지 프로파일 SHAP
    cognitive_cols = [c for c in all_features.columns if c.startswith('cognitive_')]
    if len(cognitive_cols) > 0:
        cognitive_shap_cols = [c for c in cognitive_cols if c in X_shap.columns]
        if len(cognitive_shap_cols) > 0:
            print("\n  인지 프로파일 SHAP 분석...")
            X_cognitive_shap = X_shap[cognitive_shap_cols]
            cognitive_shap_indices = [list(X_shap.columns).index(c) for c in cognitive_shap_cols]
            cognitive_shap_values = shap_values[1][:, cognitive_shap_indices]
            
            plt.figure(figsize=(10, 6))
            shap.summary_plot(cognitive_shap_values, X_cognitive_shap, plot_type="bar", show=False)
            plt.title('SHAP Feature Importance - Cognitive Profiles', fontsize=12, fontweight='bold')
            plt.tight_layout()
            plt.show()
elif HAS_SHAP and not HAS_LIGHTGBM:
    print("\n[2] SHAP 분석 스킵 (LightGBM이 필요합니다)")
    print("   SHAP은 LightGBM 모델과 함께 사용됩니다.")
else:
    print("\n[2] SHAP 분석 스킵 (라이브러리 없음)")

print("\n✅ 피처 중요도 분석 완료!")


피처 중요도 분석


FileNotFoundError: 피처 파일이 없습니다: ./driver_level_features.parquet
Cell 48을 먼저 실행하세요.

In [ ]:
# 인지 프로파일 기반 분석
import os

print("=" * 80)
print("인지 프로파일 기반 분석")
print("=" * 80)

# all_features가 없으면 저장된 파일에서 로드
if 'all_features' not in globals() or all_features is None:
    output_path = "./driver_level_features.parquet"
    if os.path.exists(output_path):
        print(f"📂 저장된 피처 파일 로드: {output_path}")
        all_features = pd.read_parquet(output_path)
        print(f"✅ 로드 완료: {all_features.shape}")
    else:
        raise FileNotFoundError(f"피처 파일이 없습니다: {output_path}\nCell 48을 먼저 실행하세요.")

# 인지 프로파일 컬럼
cognitive_cols = [c for c in all_features.columns if c.startswith('cognitive_')]
print(f"인지 프로파일 컬럼: {len(cognitive_cols)}개")
print(f"  {cognitive_cols}")

if len(cognitive_cols) == 0:
    print("인지 프로파일 컬럼이 없습니다. 인지 프로파일 생성 함수를 먼저 실행하세요.")
else:
    # ============================================================
    # 1. 인지 프로파일별 라벨 분포
    # ============================================================
    print("\n[1] 인지 프로파일별 라벨 분포 분석")
    
    X_cognitive = all_features[cognitive_cols].fillna(all_features[cognitive_cols].median())
    y_cognitive = all_features['Label'].values
    
    # 각 프로파일의 평균값 계산
    profile_stats = pd.DataFrame({
        'feature': cognitive_cols,
        'mean_label0': [X_cognitive[y_cognitive == 0][col].mean() for col in cognitive_cols],
        'mean_label1': [X_cognitive[y_cognitive == 1][col].mean() for col in cognitive_cols],
        'std_label0': [X_cognitive[y_cognitive == 0][col].std() for col in cognitive_cols],
        'std_label1': [X_cognitive[y_cognitive == 1][col].std() for col in cognitive_cols],
    })
    profile_stats['diff'] = profile_stats['mean_label1'] - profile_stats['mean_label0']
    profile_stats = profile_stats.sort_values('diff', ascending=False)
    
    print("\n인지 프로파일별 평균값 (Label 0 vs 1):")
    display(profile_stats)
    
    # 시각화
    fig, axes = plt.subplots(len(cognitive_cols), 1, figsize=(12, 4 * len(cognitive_cols)))
    if len(cognitive_cols) == 1:
        axes = [axes]
    
    for idx, col in enumerate(cognitive_cols):
        ax = axes[idx]
        label_0_data = X_cognitive[y_cognitive == 0][col].dropna()
        label_1_data = X_cognitive[y_cognitive == 1][col].dropna()
        
        ax.hist(label_0_data, alpha=0.5, label='Label 0', bins=30, color='skyblue', density=True)
        ax.hist(label_1_data, alpha=0.5, label='Label 1', bins=30, color='salmon', density=True)
        ax.set_xlabel(col)
        ax.set_ylabel('Density')
        ax.set_title(f'{col} Distribution by Label', fontsize=10, fontweight='bold')
        ax.legend()
        ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # 2. 로지스틱 회귀로 인지 프로파일 해석
    # ============================================================
    print("\n[2] 로지스틱 회귀 분석 (인지 프로파일)")
    
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    
    # 스케일링
    scaler_lr = StandardScaler()
    X_cognitive_scaled = scaler_lr.fit_transform(X_cognitive)
    
    # 로지스틱 회귀
    lr_model = LogisticRegression(
        penalty='l2',
        C=1.0,
        class_weight='balanced',  # 불균형 처리
        random_state=42,
        max_iter=1000
    )
    lr_model.fit(X_cognitive_scaled, y_cognitive)
    
    # 계수 분석
    coef_df = pd.DataFrame({
        'feature': cognitive_cols,
        'coefficient': lr_model.coef_[0],
        'abs_coefficient': np.abs(lr_model.coef_[0])
    }).sort_values('abs_coefficient', ascending=False)
    
    print("\n로지스틱 회귀 계수 (절대값 기준 정렬):")
    display(coef_df)
    
    # 계수 시각화
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ['red' if c < 0 else 'blue' for c in coef_df['coefficient']]
    ax.barh(range(len(coef_df)), coef_df['coefficient'], color=colors)
    ax.set_yticks(range(len(coef_df)))
    ax.set_yticklabels(coef_df['feature'])
    ax.set_xlabel('Coefficient')
    ax.set_title('Logistic Regression Coefficients - Cognitive Profiles', fontsize=12, fontweight='bold')
    ax.axvline(x=0, color='black', linestyle='--', linewidth=0.5)
    ax.grid(alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
    
    # ============================================================
    # 3. 인지 프로파일 조합 분석
    # ============================================================
    print("\n[3] 인지 프로파일 조합 분석")
    
    # 각 프로파일을 상/중/하로 분류
    profile_combinations = []
    
    for col in cognitive_cols:
        median_val = X_cognitive[col].median()
        q25 = X_cognitive[col].quantile(0.25)
        q75 = X_cognitive[col].quantile(0.75)
        
        # 낮음/보통/높음으로 분류
        low_mask = X_cognitive[col] <= q25
        high_mask = X_cognitive[col] >= q75
        
        profile_combinations.append({
            'feature': col,
            'low_count': low_mask.sum(),
            'high_count': high_mask.sum(),
            'low_risk_rate': y_cognitive[low_mask].mean() if low_mask.sum() > 0 else 0,
            'high_risk_rate': y_cognitive[high_mask].mean() if high_mask.sum() > 0 else 0,
        })
    
    combo_df = pd.DataFrame(profile_combinations)
    combo_df['risk_diff'] = combo_df['high_risk_rate'] - combo_df['low_risk_rate']
    combo_df = combo_df.sort_values('risk_diff', ascending=False)
    
    print("\n인지 프로파일별 위험도 차이:")
    display(combo_df)
    
    # 조합 분석: "지각운동 약함 + 시각기억 약함 + 스트레스 높음"
    print("\n[4] 특정 조합 분석 예시")
    
    # 예시: 지각운동 정확도 낮음 + 스트레스 높음
    if 'cognitive_perceptual_motor_acc' in cognitive_cols and 'cognitive_stress_level' in cognitive_cols:
        motor_low = X_cognitive['cognitive_perceptual_motor_acc'] <= X_cognitive['cognitive_perceptual_motor_acc'].quantile(0.33)
        stress_high = X_cognitive['cognitive_stress_level'] >= X_cognitive['cognitive_stress_level'].quantile(0.67)
        
        combo_mask = motor_low & stress_high
        combo_risk_rate = y_cognitive[combo_mask].mean() if combo_mask.sum() > 0 else 0
        
        print(f"  조합: 지각운동 정확도 낮음 + 스트레스 높음")
        print(f"  해당 조합 샘플 수: {combo_mask.sum()}개")
        print(f"  위험군 비율: {combo_risk_rate:.4f}")
        print(f"  전체 위험군 비율: {y_cognitive.mean():.4f}")
        print(f"  비율 차이: {combo_risk_rate - y_cognitive.mean():.4f}")
        
        # 시각화
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # 조합별 위험도
        combo_labels = ['Other', 'Motor Low + Stress High']
        combo_risk_rates = [
            y_cognitive[~combo_mask].mean() if (~combo_mask).sum() > 0 else 0,
            combo_risk_rate
        ]
        
        axes[0].bar(combo_labels, combo_risk_rates, color=['skyblue', 'salmon'])
        axes[0].set_ylabel('Risk Rate')
        axes[0].set_title('Risk Rate by Cognitive Profile Combination', fontsize=12, fontweight='bold')
        axes[0].grid(alpha=0.3, axis='y')
        axes[0].axhline(y=y_cognitive.mean(), color='red', linestyle='--', label='Overall Risk Rate')
        axes[0].legend()
        
        # 산점도
        scatter_colors = ['red' if y_cognitive[i] == 1 else 'blue' for i in range(len(y_cognitive))]
        axes[1].scatter(X_cognitive['cognitive_perceptual_motor_acc'], 
                       X_cognitive['cognitive_stress_level'],
                       c=scatter_colors, alpha=0.3, s=10)
        axes[1].axvline(x=X_cognitive['cognitive_perceptual_motor_acc'].quantile(0.33), 
                       color='green', linestyle='--', label='Motor Low Threshold')
        axes[1].axhline(y=X_cognitive['cognitive_stress_level'].quantile(0.67), 
                       color='orange', linestyle='--', label='Stress High Threshold')
        axes[1].set_xlabel('Cognitive Perceptual Motor Acc')
        axes[1].set_ylabel('Cognitive Stress Level')
        axes[1].set_title('Cognitive Profile Combination', fontsize=12, fontweight='bold')
        axes[1].legend()
        axes[1].grid(alpha=0.3)
        
        plt.tight_layout()
        plt.show()
    
    print("\n✅ 인지 프로파일 기반 분석 완료!")


인지 프로파일 기반 분석


FileNotFoundError: 피처 파일이 없습니다: ./driver_level_features.parquet
Cell 48을 먼저 실행하세요.

## 19. 요약 및 다음 단계

분석 결과 요약 및 모델 개선 방향
